In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H20 — Deep Escalation + Basin Clustering + Fused Extraction
# ══════════════════════════════════════════════════════════════════════
#
# COMBINES:
#   H15 (depth):  Escalating rounds R1-R5, high particle counts,
#                 progressive noise for basin escape
#   H19 (breadth): K engines per round, basin clustering,
#                  fused GPU extraction
#
# ARCHITECTURE (per instance):
#   R1: K engines × P particles × 4000 steps (diverse seeds)
#       → fused extract top-3/engine → basin cluster → FULL BPR
#   R2-R5: escalating re-ignition (seed from global best)
#       → K engines × seeded gravity × 2500-3000 steps
#       → fused extract top-3/engine → basin cluster → FULL BPR
#       Escalation: seed_frac ↓ (50→25%), noise ↑ (0.30→0.60)
#
# KEY DESIGN:
#   • K=8 engines × P=1000 particles → MATCHES H15 depth per engine
#   • Top-3 per engine → 24 candidates → basin cluster → ≤8 basins
#   • Basin clustering: Hamming τ=15%n, max 8 basins
#   • BPR: 2 attempts per basin, full flip budget (no tiers!)
#     → 8 basins × 2 = 16 BPR attempts ≈ H15's effective ~15
#   • Clean gravity core (no anti-collapse/micro-reset — H15 proven)
#   • Diverse seeding in R1 (5 types), escalating seed in R2-R5
#   • Drop R6-R8 (σ≥0.7 proven dead in H15 data)
#
# FIXES vs H20v1:
#   • Removed BPR tiers — they starved hard cases at α=4.2
#     (20K flips/30% prob → always full budget now)
#   • 2 BPR attempts per basin center (diverse random walks)
#     → 16 attempts/round ≈ H15's 15, from 8 clustered starts
#   • Removed anti-collapse@50 + micro-reset@200 (from H19, which
#     was a disaster; H15 never had them and works fine)
#   • 1000 ptcl per engine (H20v1 ran 750 on Colab = 25% shallower)
#   • MAX_BASINS 6→8 for more BPR diversity
#   • Per-engine noise diversity in R2-R5 seeded rounds
#     (engine 0 → 1× noise, engine K-1 → 3× noise)
#     Fixes R2-R5 basin collapse: all engines converged to 1 basin
#     because they all seeded from the same best with same noise.
#     Now engines span focused→exploratory, producing ≥3 basins.
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time, math, os
from numba import njit
from concurrent.futures import ThreadPoolExecutor, as_completed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

N_WORKERS = min(os.cpu_count() or 2, 8)

# ═══════ OPT [E]: Auto-detect AMP dtype (T4=fp16, A100/H100=bf16) ═══════
if device.type == 'cuda':
    _amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    _amp_dtype = torch.float32

# ═══════ OPT [F]: Global BPR thread pool — reuse across all rounds ═══════
_BPR_POOL = ThreadPoolExecutor(max_workers=N_WORKERS)

# ═══════ AUTO-SCALE BASED ON GPU VRAM ═══════
def _get_vram_gb():
    p = torch.cuda.get_device_properties(0)
    return getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9

if device.type == 'cuda':
    _vram = _get_vram_gb()
    if _vram >= 70:      # A100-80GB / RTX PRO 6000 (102GB)
        K_ENGINES = 8;  GRAVITY_BATCH = 16;  P_R1 = 1000;  P_RN = 1000
    elif _vram >= 35:    # A100-40GB
        K_ENGINES = 6;  GRAVITY_BATCH = 12;  P_R1 = 1000;  P_RN = 800
    elif _vram >= 14:    # T4 / V100
        K_ENGINES = 4;  GRAVITY_BATCH = 8;   P_R1 = 800;   P_RN = 600
    else:
        K_ENGINES = 3;  GRAVITY_BATCH = 6;   P_R1 = 600;   P_RN = 500
else:
    K_ENGINES = 2;  GRAVITY_BATCH = 4;  P_R1 = 500;  P_RN = 400

# ═══════ ESCALATION SCHEDULE (R2-R5) ═══════
ESCALATION = [
    # R2: moderate — 50% seed near best, σ=0.30
    {'seed_frac': 0.50, 'noise': 0.30, 'steps': 2500, 'flips': 200000},
    # R3: wider — 40% seed, noise grows
    {'seed_frac': 0.40, 'noise': 0.40, 'steps': 2500, 'flips': 200000},
    # R4: aggressive — 30% seed, big noise
    {'seed_frac': 0.30, 'noise': 0.50, 'steps': 3000, 'flips': 250000},
    # R5: very aggressive — 25% seed, σ=0.60, longer gravity
    {'seed_frac': 0.25, 'noise': 0.60, 'steps': 3000, 'flips': 300000},
]
MAX_ROUNDS = len(ESCALATION) + 1  # +1 for R1

# ═══════ H20 ARCHITECTURE PARAMETERS ═══════
STEPS_R1       = 4000     # R1 gravity depth
FLIPS_R1       = 200000   # R1 BPR budget

# Basin clustering
MAX_BASINS         = 8     # was 6 → more diverse BPR starts
TOP_CANDIDATES     = 32
UNSAT_REJECT_FRAC  = 0.20
HAMMING_TAU_FRAC   = 0.15

# BPR — FULL budget, multiple attempts per basin (no tiers!)
N_BPR_PER_BASIN = 2   # run 2 BPR workers per basin center
                       # → 8 basins × 2 = 16 attempts ≈ H15's ~15

# Extraction
TOP_K_PER_ENGINE = 3   # extract top-3 particles per engine for basin clustering

# BPR chain parameters
CHAIN_PATIENCE  = 5000
BRANCH_PATIENCE = 80
COOL_RATE       = 0.95
STUB_FRAC       = 0.08
WEIGHT_BUMP     = 2.0
WEIGHT_DECAY    = 0.9
BPR_BETA        = 0.3


# ══════════════════════════════════════════════════════════════════════
#  INSTANCE GENERATOR
# ══════════════════════════════════════════════════════════════════════

def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  TREE-WALK BPR — chain death + stubbornness
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_chain(clauses_v, clauses_s, assignment, weight,
              max_flips=200000, T_init=0.5, T_min=0.01,
              p_random=0.1, beta=0.3,
              chain_patience=5000, branch_patience=80,
              cool_rate=0.95, stub_frac=0.08,
              weight_bump=2.0, weight_decay=0.9):
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    n_stub = max(2, int(stub_frac * n))

    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    clause_w = np.ones(m, dtype=np.float64)
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    best_n_unsat = n_unsat
    best_assign = assignment.copy()
    chain_id = 0
    chain_stale = 0
    chain_best_unsat = n_unsat
    in_branch = False
    branch_stale = 0
    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0
    stubbornness = np.zeros(n, dtype=np.float64)

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip, chain_id + 1, 0

        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]
            chain_stale = 0
            chain_best_unsat = n_unsat
        elif n_unsat < chain_best_unsat:
            chain_best_unsat = n_unsat
            chain_stale = 0
        else:
            chain_stale += 1

        if chain_stale >= chain_patience and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += weight_bump
            for c in range(m):
                clause_w[c] *= weight_decay
            for vi in range(n):
                stubbornness[vi] = 0.0
            for ui in range(n_unsat):
                cc = unsat_list[ui]
                w_c = clause_w[cc]
                for j in range(3):
                    stubbornness[clauses_v[cc, j]] += w_c
            stub_to_flip = np.zeros(n_stub, dtype=np.int32)
            stub_used = np.zeros(n, dtype=np.int8)
            for k in range(n_stub):
                best_sv = -1.0
                best_vi = 0
                for vi in range(n):
                    if stub_used[vi] == 0 and stubbornness[vi] > best_sv:
                        best_sv = stubbornness[vi]
                        best_vi = vi
                stub_to_flip[k] = best_vi
                stub_used[best_vi] = 1
            for i in range(n):
                assignment[i] = best_assign[i]
            for k in range(n_stub):
                assignment[stub_to_flip[k]] = 1 - assignment[stub_to_flip[k]]
            n_random = max(1, n // 100)
            for _ in range(n_random):
                vi = np.random.randint(n)
                if np.random.random() < 0.3:
                    assignment[vi] = 1 - assignment[vi]

            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1

            if n_unsat == 0:
                return assignment, flip, chain_id + 1, 0
            chain_id += 1
            chain_stale = 0
            chain_best_unsat = n_unsat
            in_branch = False
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0
            continue

        ci = -1
        if in_branch and branch_stale < branch_patience:
            best_neighbor_w = -1.0
            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc
            if ci < 0:
                in_branch = False

        if not in_branch or ci < 0:
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        T_max_chain = T_init * (cool_rate ** min(chain_id, 30))
        local_prog = min(1.0, chain_stale / chain_patience)
        T = T_min + (T_max_chain - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)
            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0; w_brk = 0.0; w_make = 0.0
                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]
                int_brks[j] = i_brk
                delta = w_brk - w_make
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1
            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]
        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        if n_unsat < old_n_unsat:
            branch_stale = 0
        else:
            branch_stale += 1
        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips, chain_id + 1, best_n_unsat


# ══════════════════════════════════════════════════════════════════════
#  NUMBA SAT / UNSAT
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def check_sat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            return False
    return True

@njit(cache=True)
def count_unsat(clauses_v, clauses_s, assignment):
    m = clauses_v.shape[0]
    cnt = 0
    for c in range(m):
        sat = False
        for j in range(3):
            vi = clauses_v[c, j]
            si = clauses_s[c, j]
            if (assignment[vi] == 1 and si == 1) or (assignment[vi] == 0 and si == -1):
                sat = True
                break
        if not sat:
            cnt += 1
    return cnt


# ══════════════════════════════════════════════════════════════════════
#  BATCHED ENERGY — unrolled j-loop, in-place product accumulation
# ══════════════════════════════════════════════════════════════════════

def _energy_batched(s, mu_val, vars_t, signs_t):
    B, P, n = s.shape
    m = vars_t.shape[1]
    v0 = vars_t[:, :, 0].unsqueeze(1).expand(B, P, m)
    v1 = vars_t[:, :, 1].unsqueeze(1).expand(B, P, m)
    v2 = vars_t[:, :, 2].unsqueeze(1).expand(B, P, m)
    s0 = signs_t[:, None, :, 0]
    s1 = signs_t[:, None, :, 1]
    s2 = signs_t[:, None, :, 2]
    prod = 1.0 - torch.gather(s, 2, v0) * s0
    prod *= (1.0 - torch.gather(s, 2, v1) * s1)
    prod *= (1.0 - torch.gather(s, 2, v2) * s2)
    e_sat = prod.sum(dim=-1) / 8.0
    if mu_val > 0:
        return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(dim=-1)
    return e_sat


# ══════════════════════════════════════════════════════════════════════
#  GRAVITY CORE — CLEAN (matches H15 proven design)
#  NO anti-collapse, NO micro-reset (removed: H19 features that hurt)
#  [A] Pre-allocated scalar tensors
#  [B] Batched elite gravity (no Python for-loop over B)
#  [C] torch.autograd.grad (no .backward() + .grad.clone())
# ══════════════════════════════════════════════════════════════════════

def _gravity_core(s, vars_t, signs_t, steps, lr=0.05,
                  momentum_beta=0.9, mu_scale=0.1,
                  G_max=0.10, top_k_frac=0.1,
                  gravity_start=0.2, elite_repulsion=0.5,
                  gravity_interval=20):
    B, particles, n = s.shape
    gi = gravity_interval
    use_amp = (device.type == 'cuda')

    vel = torch.zeros_like(s)
    grav_step = int(gravity_start * steps)
    delay_step = int(0.7 * steps)
    top_k = max(1, int(top_k_frac * particles))
    theta = None

    best_e = torch.full((B, particles), float('inf'), device=device)
    plateau_count = torch.zeros(B, particles, device=device)
    decay_arr = 1.0 / (1.0 + 0.002 * torch.arange(
        steps, device=device, dtype=torch.float32))

    # OPT [A]: Pre-allocate scalar constants
    _scalar_2 = torch.tensor(2.0, device=device)
    _scalar_1 = torch.tensor(1.0, device=device)
    _scalar_4 = torch.tensor(4.0, device=device)

    # OPT [B]: Pre-compute eye mask for elite repulsion
    cached_targets_t = None
    if top_k > 1:
        _eye_mask = torch.eye(top_k, device=device, dtype=torch.bool).unsqueeze(0)

    for step in range(steps):
        if step < delay_step:
            mu_val = 0.0
        else:
            t_l = (step - delay_step) / (steps - delay_step)
            mu_val = mu_scale * 0.5 * (1.0 - math.cos(math.pi * t_l))

        s = s.detach().requires_grad_(True)
        if use_amp:
            with torch.amp.autocast('cuda', dtype=_amp_dtype):
                e = _energy_batched(s, mu_val, vars_t, signs_t)
                e_f32 = e.float()
        else:
            e_f32 = _energy_batched(s, mu_val, vars_t, signs_t)

        e_vals = e_f32.detach()

        # OPT [C]: torch.autograd.grad
        (g,) = torch.autograd.grad(e_f32.sum(), s)
        g = g.detach()

        with torch.no_grad():
            decay = decay_arr[step]
            improved = e_vals < best_e
            best_e = torch.where(improved, e_vals, best_e)
            plateau_count = torch.where(improved,
                torch.zeros_like(plateau_count), plateau_count + 1)
            pm = (plateau_count >= 50).unsqueeze(2)

            gnorm = g.norm(dim=2, keepdim=True).clamp_(min=1e-10)
            dte = (lr * decay) / (1.0 + 0.05 * gnorm)
            dte = dte * torch.where(pm, _scalar_2, _scalar_1)

            if theta is None:
                theta = float(e_vals.median()) + 1e-8
            damp = 1.0 / (1.0 + e_vals.unsqueeze(2) / theta)
            gam = (e_vals.clamp(min=0) / (e_vals + 1.0)).unsqueeze(2)

            # In-place velocity update
            step_scale = dte * damp
            step_scale *= (1.0 + gam)
            vel.mul_(momentum_beta).sub_(step_scale * g)

            # In-place position update
            ns_base = 0.03 * decay
            noise_scale = torch.where(pm, _scalar_4 * ns_base, ns_base)
            s = s.detach()
            s.add_(vel)
            s.addcmul_(torch.randn_like(s), noise_scale, value=1.0)
            s.clamp_(-1, 1)

            # ═══ OPT [B]: Elite gravity — FULLY BATCHED ═══
            if step >= grav_step and (step - grav_step) % gi == 0:
                progress = (step - grav_step) / (steps - grav_step)
                g_mag = G_max * progress * progress * gi

                _, all_top_idx = e_vals.topk(top_k, dim=1, largest=False)
                tk_exp = all_top_idx.unsqueeze(2).expand(B, top_k, n)
                all_elite = torch.gather(s, 1, tk_exp)

                d_all = torch.cdist(s, all_elite)
                nn_idx = d_all.argmin(dim=2)
                nn_exp = nn_idx.unsqueeze(2).expand(B, particles, n)
                cached_targets_t = torch.gather(all_elite, 1, nn_exp)

                if top_k > 1:
                    ed = torch.cdist(all_elite, all_elite)
                    ed.masked_fill_(_eye_mask, float('inf'))
                    nn_e_idx = ed.argmin(dim=2)
                    nn_e_exp = nn_e_idx.unsqueeze(2).expand(B, top_k, n)
                    nearest_elite = torch.gather(all_elite, 1, nn_e_exp)
                    push = all_elite - nearest_elite
                    pn = push.norm(dim=2, keepdim=True).clamp_(min=1e-6)
                    s.scatter_add_(1, tk_exp, (elite_repulsion * g_mag) * (push / pn))

                s.add_(g_mag * (cached_targets_t - s))

            elif step >= grav_step and cached_targets_t is not None:
                progress = (step - grav_step) / (steps - grav_step)
                g_mag = G_max * progress * progress
                s.add_(g_mag * (cached_targets_t - s))

            s.clamp_(-1, 1)
            if (step + 1) % 200 == 0:
                theta = float(e_vals.median()) + 1e-8

    return s.detach()


# ══════════════════════════════════════════════════════════════════════
#  DIVERSE SEED GENERATION (5 types — for R1)
# ══════════════════════════════════════════════════════════════════════

def generate_diverse_seeds(K, n, best_model=None):
    seeds = np.zeros((K, n), dtype=np.int32)
    if best_model is None:
        for k in range(K):
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)
        return seeds

    for k in range(K):
        r = np.random.random()
        if r < 0.30:
            seeds[k] = best_model.copy()
            nf = max(1, int(0.05 * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        elif r < 0.50:
            seeds[k] = 1 - best_model
        elif r < 0.70:
            seeds[k] = best_model.copy()
            frac = 0.20 + 0.10 * np.random.random()
            nf = max(1, int(frac * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        elif r < 0.90:
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)
        else:
            seeds[k] = best_model.copy()
            bsz = n // 4
            start = np.random.randint(0, n - bsz + 1)
            seeds[k][start:start + bsz] = 1 - seeds[k][start:start + bsz]
    return seeds


# ══════════════════════════════════════════════════════════════════════
#  BASIN CLUSTERING — Hamming distance (Numba)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def basin_cluster_numba(cand_bin, cand_unsat, n, tau, max_clusters):
    K = cand_bin.shape[0]
    order = np.argsort(cand_unsat)
    centers = np.zeros(max_clusters, dtype=np.int32)
    n_clusters = 0

    for oi in range(K):
        i = order[oi]
        near = False
        for c in range(n_clusters):
            ci = centers[c]
            dist = 0
            for j in range(n):
                if cand_bin[i, j] != cand_bin[ci, j]:
                    dist += 1
                    if dist > tau:
                        break
            if dist <= tau:
                near = True
                break
        if not near and n_clusters < max_clusters:
            centers[n_clusters] = i
            n_clusters += 1

    return centers, n_clusters


# ══════════════════════════════════════════════════════════════════════
#  INSTANCE HELPER
# ══════════════════════════════════════════════════════════════════════

class InstanceHelper:
    def __init__(self, n, clauses):
        self.n = n
        self.m = len(clauses)
        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t = torch.tensor(vs_list, dtype=torch.long, device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)


# ══════════════════════════════════════════════════════════════════════
#  BPR WORKER
# ══════════════════════════════════════════════════════════════════════

def bpr_worker(clauses_v, clauses_s, x_np, weight,
               max_flips, T_init, T_min, p_random, beta,
               chain_patience, branch_patience,
               cool_rate, stub_frac, weight_bump, weight_decay):
    sol, flips, n_chains, remaining = bpr_chain(
        clauses_v, clauses_s, x_np, weight,
        max_flips=max_flips, T_init=T_init, T_min=T_min,
        p_random=p_random, beta=beta,
        chain_patience=chain_patience, branch_patience=branch_patience,
        cool_rate=cool_rate, stub_frac=stub_frac,
        weight_bump=weight_bump, weight_decay=weight_decay
    )
    is_sat = check_sat(clauses_v, clauses_s, sol)
    return is_sat, flips, n_chains, sol


# ══════════════════════════════════════════════════════════════════════
#  FUSED GPU CANDIDATE EXTRACTION — top-K per engine
# ══════════════════════════════════════════════════════════════════════

def _fused_extract_candidates(s_final, vars_t, signs_t, pos_mask, top_k=3):
    """
    GPU-side: discretize all particles, find top-K per engine, extract.
    Returns list of B lists, each containing top_k dicts.
    """
    B, P, n = s_final.shape
    top_k = min(top_k, P)
    with torch.no_grad():
        x_bin = (s_final > 0).long()

        m_cls = vars_t.shape[1]
        clause_sat = torch.zeros(B, P, m_cls, device=s_final.device, dtype=torch.bool)
        for j in range(3):
            idx_j = vars_t[:, :, j].unsqueeze(1).expand(B, P, m_cls)
            gathered = torch.gather(x_bin, 2, idx_j)
            pm_j = pos_mask[:, :, j].unsqueeze(1).expand(B, P, m_cls)
            clause_sat = clause_sat | (gathered == pm_j)
        n_sat = clause_sat.sum(dim=2)

        topk_sat, topk_idx = n_sat.topk(top_k, dim=1, largest=True)
        topk_unsat = m_cls - topk_sat

        b_range = torch.arange(B, device=s_final.device).unsqueeze(1).expand(B, top_k)
        topk_assignments = x_bin[b_range, topk_idx]
        topk_conf = 1.0 - s_final[b_range, topk_idx].abs()

        assignments_np = topk_assignments.cpu().numpy().astype(np.int32)
        unsats_np = topk_unsat.cpu().numpy().astype(np.int32)
        conf_np = topk_conf.cpu().numpy().astype(np.float64)

    results = []
    for b in range(B):
        engine_cands = []
        for k in range(top_k):
            engine_cands.append({
                'assignment': assignments_np[b, k],
                'unsat': int(unsats_np[b, k]),
                'confidence': conf_np[b, k],
            })
        results.append(engine_cands)
    return results


# ══════════════════════════════════════════════════════════════════════
#  RUN ROUND GRAVITY — K engines, deep particles, fused extraction
# ══════════════════════════════════════════════════════════════════════

def run_round_gravity(unsolved_helpers, seeds_per_inst, n, m,
                      particles, steps, batch_size):
    N = len(unsolved_helpers)
    K = seeds_per_inst[0].shape[0]
    total = N * K

    flat_helper_idx = []
    flat_seeds = np.zeros((total, n), dtype=np.int32)
    for i in range(N):
        for k in range(K):
            flat_helper_idx.append(i)
            flat_seeds[i * K + k] = seeds_per_inst[i][k]

    all_candidates = [None] * total

    for bs in range(0, total, batch_size):
        be = min(bs + batch_size, total)
        B = be - bs

        vars_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].vars_t
            for e in range(bs, be)
        ])
        signs_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].signs_t
            for e in range(bs, be)
        ])
        pos_mask_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].pos_mask
            for e in range(bs, be)
        ])

        s_seeds = torch.tensor(
            flat_seeds[bs:be], dtype=torch.float32, device=device
        )
        s_seeds = s_seeds * 1.4 - 0.7
        s = s_seeds.unsqueeze(1).expand(B, particles, n).clone()
        s += torch.randn(B, particles, n, device=device) * 0.3
        s.clamp_(-0.9, 0.9)

        s_final = _gravity_core(s, vars_batch, signs_batch, steps=steps)

        batch_candidates = _fused_extract_candidates(
            s_final, vars_batch, signs_batch, pos_mask_batch,
            top_k=TOP_K_PER_ENGINE
        )
        for b in range(B):
            all_candidates[bs + b] = batch_candidates[b]

        del s, s_final, vars_batch, signs_batch, pos_mask_batch

    result = []
    for i in range(N):
        inst_cands = []
        for k in range(K):
            inst_cands.extend(all_candidates[i * K + k])
        result.append(inst_cands)
    return result


# ══════════════════════════════════════════════════════════════════════
#  RUN ROUND SEEDED GRAVITY — escalating noise, K engines
#  OPT [D]: Vectorized seed construction
#  FIX [H20v3]: Per-engine noise diversity to prevent R2-R5 basin
#               collapse — engine 0 is focused (1× noise), engine K-1
#               is exploratory (3× noise).  This ensures basin
#               clustering discovers ≥3 distinct basins instead of 1.
# ══════════════════════════════════════════════════════════════════════

def run_round_seeded_gravity(unsolved_helpers, global_bests, n, m,
                             particles, steps, seed_frac, seed_noise,
                             batch_size):
    N = len(unsolved_helpers)
    total = N * K_ENGINES
    n_seeded = int(seed_frac * particles)
    n_random = particles - n_seeded

    all_candidates = [None] * total

    flat_helper_idx = []
    flat_global_best = []
    flat_engine_k = []  # track engine index for noise diversity
    for i in range(N):
        for k in range(K_ENGINES):
            flat_helper_idx.append(i)
            flat_global_best.append(global_bests[i])
            flat_engine_k.append(k)

    for bs in range(0, total, batch_size):
        be = min(bs + batch_size, total)
        B = be - bs

        vars_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].vars_t
            for e in range(bs, be)
        ])
        signs_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].signs_t
            for e in range(bs, be)
        ])
        pos_mask_batch = torch.stack([
            unsolved_helpers[flat_helper_idx[e]].pos_mask
            for e in range(bs, be)
        ])

        # OPT [D]: Vectorized seed construction
        gb_batch = np.stack([flat_global_best[bs + b] for b in range(B)])
        s_seed = torch.from_numpy(gb_batch).float().to(device) * 1.4 - 0.7

        s_hot = s_seed.unsqueeze(1).expand(B, n_seeded, n).clone()

        # FIX [H20v3]: Per-engine noise diversity
        # Engine 0 → 1.0× noise (focused near best-known solution)
        # Engine K-1 → 3.0× noise (exploratory, nearly random starts)
        # This prevents all engines converging to one basin in R2-R5
        Km1 = max(1, K_ENGINES - 1)
        noise_mult = torch.tensor(
            [1.0 + 2.0 * flat_engine_k[bs + b] / Km1 for b in range(B)],
            device=device, dtype=torch.float32
        ).view(B, 1, 1)
        s_hot += torch.randn(B, n_seeded, n, device=device) * seed_noise * noise_mult
        s_hot.clamp_(-0.9, 0.9)

        s_cold = (torch.randn(B, n_random, n, device=device) * 0.3).clamp_(-0.9, 0.9)

        s = torch.cat([s_hot, s_cold], dim=1)

        s_final = _gravity_core(s, vars_batch, signs_batch, steps=steps)

        batch_candidates = _fused_extract_candidates(
            s_final, vars_batch, signs_batch, pos_mask_batch,
            top_k=TOP_K_PER_ENGINE
        )
        for b in range(B):
            all_candidates[bs + b] = batch_candidates[b]

        del s, s_hot, s_cold, s_final, vars_batch, signs_batch, pos_mask_batch

    result = []
    for i in range(N):
        inst_cands = []
        for k in range(K_ENGINES):
            inst_cands.extend(all_candidates[i * K_ENGINES + k])
        result.append(inst_cands)
    return result


# ══════════════════════════════════════════════════════════════════════
#  PROCESS ONE INSTANCE IN A ROUND
#  - FULL BPR budget for every basin (no tiers!)
#  - N_BPR_PER_BASIN attempts per basin (diverse random walks)
#  - Global thread pool + early cancellation
# ══════════════════════════════════════════════════════════════════════

def process_instance_round(helper, candidates_list, best_model,
                           best_unsat, n, bpr_max_flips):
    m = helper.m
    candidates = candidates_list

    # Check gravity-only solves
    for c in candidates:
        if c['unsat'] == 0:
            return True, c['assignment'], 0, 0, 0, True

    # Stage 1: reject if unsat > 20% × n
    threshold = int(UNSAT_REJECT_FRAC * n)
    candidates = [c for c in candidates if c['unsat'] <= threshold]

    if not candidates:
        return False, best_model, best_unsat, 0, 0, False

    # Stage 2: sort by unsat, keep top
    candidates.sort(key=lambda c: c['unsat'])
    candidates = candidates[:TOP_CANDIDATES]

    # Basin clustering
    cand_bin = np.array([c['assignment'] for c in candidates], dtype=np.int32)
    cand_unsat = np.array([c['unsat'] for c in candidates], dtype=np.int32)
    tau = int(HAMMING_TAU_FRAC * n)

    centers, n_clusters = basin_cluster_numba(
        cand_bin, cand_unsat, n, tau, MAX_BASINS
    )
    basin_indices = [int(centers[c]) for c in range(n_clusters)]

    n_bpr_ran = 0
    solved = False
    new_best_model = best_model
    new_best_unsat = best_unsat

    if not basin_indices:
        return False, best_model, best_unsat, 0, 0, False

    # Submit N_BPR_PER_BASIN workers per basin, each with full flip budget
    futures = {}
    for bi in basin_indices:
        cand = candidates[bi]
        for attempt in range(N_BPR_PER_BASIN):
            f = _BPR_POOL.submit(
                bpr_worker, helper.clauses_v, helper.clauses_s,
                cand['assignment'].copy(), cand['confidence'],
                bpr_max_flips, 0.5, 0.01, 0.1, BPR_BETA,
                CHAIN_PATIENCE, BRANCH_PATIENCE,
                COOL_RATE, STUB_FRAC, WEIGHT_BUMP, WEIGHT_DECAY
            )
            futures[f] = bi
            n_bpr_ran += 1

    for f in as_completed(futures):
        is_sat, flips_used, n_chains, sol = f.result()
        sol_unsat = 0 if is_sat else count_unsat(
            helper.clauses_v, helper.clauses_s, sol
        )

        if is_sat:
            solved = True
            new_best_model = sol
            new_best_unsat = 0
            # Early cancel remaining
            for remaining in futures:
                if remaining is not f:
                    remaining.cancel()
            break
        elif sol_unsat < new_best_unsat:
            new_best_unsat = sol_unsat
            new_best_model = sol.copy()

    return solved, new_best_model, new_best_unsat, n_clusters, n_bpr_ran, False


# ══════════════════════════════════════════════════════════════════════
#  COMPILE + WARMUP
# ══════════════════════════════════════════════════════════════════════

_wv = np.array([[0, 1, 2]], dtype=np.int32)
_ws = np.array([[1, -1, 1]], dtype=np.int32)
_wa = np.array([1, 0, 1], dtype=np.int32)
_ww = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_chain(_wv, _ws, _wa, _ww, max_flips=100,
              chain_patience=20, branch_patience=5)
_ = check_sat(_wv, _ws, _wa)
_ = count_unsat(_wv, _ws, _wa)
_cb = np.array([[1, 0, 1], [0, 1, 0]], dtype=np.int32)
_cu = np.array([1, 2], dtype=np.int32)
_ = basin_cluster_numba(_cb, _cu, 3, 1, 2)

print(f'CPU cores: {os.cpu_count()}, BPR workers: {N_WORKERS}')
print()
print('✓ H20 — Deep Escalation + Basin Clustering + Fused Extraction')
print(f'  Device:  {device}')
if device.type == 'cuda':
    print(f'  GPU:     {torch.cuda.get_device_name()}')
    print(f'  VRAM:    {_get_vram_gb():.1f} GB')
    print(f'  AMP:     {_amp_dtype}')
print()
print(f'  K = {K_ENGINES} engines × top-{TOP_K_PER_ENGINE} per engine')
print(f'  R1: {STEPS_R1} steps × {P_R1} ptcl/engine (diverse seeds)')
print(f'       → {K_ENGINES}×{TOP_K_PER_ENGINE}={K_ENGINES*TOP_K_PER_ENGINE} cand → cluster(≤{MAX_BASINS}) → {MAX_BASINS}×{N_BPR_PER_BASIN}={MAX_BASINS*N_BPR_PER_BASIN} BPR @ {FLIPS_R1//1000}K')
for i, esc in enumerate(ESCALATION):
    rnd = i + 2
    print(f'  R{rnd}: {esc["steps"]} steps × {P_RN} ptcl/engine '
          f'({int(esc["seed_frac"]*100)}% seed, σ={esc["noise"]:.1f}, engine noise 1.0×–3.0×)')
    print(f'       → {K_ENGINES}×{TOP_K_PER_ENGINE}={K_ENGINES*TOP_K_PER_ENGINE} cand → cluster(≤{MAX_BASINS}) → {MAX_BASINS}×{N_BPR_PER_BASIN}={MAX_BASINS*N_BPR_PER_BASIN} BPR @ {esc["flips"]//1000}K')
print()
print(f'  Basin: τ={HAMMING_TAU_FRAC*100:.0f}%n, max {MAX_BASINS} basins')
print(f'  BPR: {N_BPR_PER_BASIN} attempts/basin, FULL flip budget (no tiers)')
print()
print('  Fixes vs H20v1:')
print('    • BPR tiers REMOVED — all basins get full flip budget')
print(f'    • {N_BPR_PER_BASIN} BPR per basin → {MAX_BASINS}×{N_BPR_PER_BASIN}={MAX_BASINS*N_BPR_PER_BASIN} attempts (was 6)')
print(f'    • MAX_BASINS 6→{MAX_BASINS}')
print(f'    • Particles {750}→{P_R1} per engine (matches H15 depth)')
print('    • Anti-collapse + micro-reset REMOVED (H19 features that hurt)')
print('    • Per-engine noise diversity in R2-R5 (1×–3× gradient)')
print('      → prevents basin collapse to 1 (was: all engines same noise → 1 basin)')
print()
print('  GPU optimizations:')
print('    [A] Pre-alloc scalars  [B] Batched elite  [C] autograd.grad')
print('    [D] Vectorized seeds   [E] Auto AMP       [F] Global BPR pool')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H20 Experiment: Deep Escalation + Basin Clustering
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50

# All baselines
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h11b_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 68.0,  (4.0, 750): 76.0,  (4.0, 1000): 52.0,
    (4.2, 500):  8.0,  (4.2, 750):  2.0,  (4.2, 1000):  2.0,
}
h13_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 94.0,  (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 30.0,  (4.2, 750): 32.0,  (4.2, 1000): 16.0,
}
h14_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 42.0,  (4.2, 750): 28.0,  (4.2, 1000): 20.0,
}
h15_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 100.0,
    (4.2, 500): 54.0,  (4.2, 750): 36.0,  (4.2, 1000): 24.0,
}
h19 = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 92.0,  (4.0, 750): 96.0,  (4.0, 1000): 80.0,
    (4.2, 500): 22.0,  (4.2, 750): 14.0,  (4.2, 1000):  2.0,
}

results = {}
round_stats = {}

print('=' * 140)
print('H20 — Deep Escalation + Basin Clustering + Fused Extraction')
print(f'  K={K_ENGINES} engines | R1: {STEPS_R1} steps × {P_R1} ptcl | '
      f'R2-R5: escalating seed+noise × {P_RN} ptcl')
print(f'  Basin τ={HAMMING_TAU_FRAC*100:.0f}%n | Max {MAX_BASINS} basins | '
      f'BPR: selective (200K/150K/80K/20K by proximity)')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'Solved':>6} | {'GravO':>5} | "
      f"{'Rnds':>4} | {'Basins':>6} | {'BPRs':>5} | "
      f"{'H15':>5} | {'H19':>5} | {'H14':>5} | {'H13':>5} | "
      f"{'ΔvsH15':>7} | Time")
print('  ' + '-' * 120)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        # ═══════════════════════════════════════════════════════════
        # Generate all instances
        # ═══════════════════════════════════════════════════════════
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        helpers = [InstanceHelper(n_var, all_instances[i]) for i in range(N_INST)]

        # ═══════════════════════════════════════════════════════════
        # TRACKING
        # ═══════════════════════════════════════════════════════════
        inst_solved = [False] * N_INST
        inst_round_solved = [0] * N_INST
        inst_best_model = [None] * N_INST
        inst_best_unsat = [m_cls] * N_INST
        inst_gravity_only = [False] * N_INST

        total_grav_only = 0
        total_basins_found = 0
        total_bpr_ran = 0
        per_round_solves = []

        # ═══════════════════════════════════════════════════════════
        # ROUND 1: K engines × deep gravity (random+diverse seeds)
        # ═══════════════════════════════════════════════════════════
        t_r1 = time.time()
        print(f'    α={alpha}, n={n_var}: R1 — {N_INST} instances, '
              f'K={K_ENGINES} engines × {P_R1} ptcl × {STEPS_R1} steps')

        # Generate diverse seeds for R1 (no best_model yet → all random)
        seeds_list = [generate_diverse_seeds(K_ENGINES, n_var, None)
                      for _ in range(N_INST)]

        engine_results = run_round_gravity(
            helpers, seeds_list, n_var, m_cls,
            particles=P_R1, steps=STEPS_R1,
            batch_size=GRAVITY_BATCH
        )
        t_grav_r1 = time.time() - t_r1

        r1_solves = 0
        r1_basins = 0
        r1_bprs = 0
        for inst in range(N_INST):
            solved, new_model, new_unsat, n_basins, n_bpr, grav_only = \
                process_instance_round(
                    helpers[inst], engine_results[inst],
                    inst_best_model[inst], inst_best_unsat[inst],
                    n_var, FLIPS_R1
                )
            r1_basins += n_basins
            r1_bprs += n_bpr

            if solved:
                inst_solved[inst] = True
                inst_round_solved[inst] = 1
                inst_best_model[inst] = new_model
                inst_best_unsat[inst] = 0
                r1_solves += 1
                if grav_only:
                    inst_gravity_only[inst] = True
                    total_grav_only += 1
            else:
                inst_best_model[inst] = new_model if new_model is not None else inst_best_model[inst]
                inst_best_unsat[inst] = new_unsat

        total_basins_found += r1_basins
        total_bpr_ran += r1_bprs
        per_round_solves.append(r1_solves)
        total_solved = sum(inst_solved)

        print(f'      → R1: +{r1_solves} solved ({total_solved}/{N_INST}), '
              f'{r1_basins} basins, {r1_bprs} BPRs | '
              f'grav={t_grav_r1:.0f}s, total={time.time()-t_r1:.0f}s')

        # ═══════════════════════════════════════════════════════════
        # ROUNDS 2-5: Escalating re-ignition
        # ═══════════════════════════════════════════════════════════
        for rnd_idx, esc in enumerate(ESCALATION):
            rnd = rnd_idx + 2

            unsolved_idx = [i for i in range(N_INST) if not inst_solved[i]]
            if not unsolved_idx:
                per_round_solves.append(0)
                continue

            sf = esc['seed_frac']
            sn = esc['noise']
            steps_rn = esc['steps']
            flips_rn = esc['flips']

            t_rn = time.time()
            print(f'    α={alpha}, n={n_var}: R{rnd} — {len(unsolved_idx)} unsolved, '
                  f'K={K_ENGINES} engines × {P_RN} ptcl × {steps_rn} steps '
                  f'({int(sf*100)}% seed, σ={sn:.1f})')

            unsolved_helpers = [helpers[i] for i in unsolved_idx]
            global_bests = [inst_best_model[i] for i in unsolved_idx]

            engine_results = run_round_seeded_gravity(
                unsolved_helpers, global_bests, n_var, m_cls,
                particles=P_RN, steps=steps_rn,
                seed_frac=sf, seed_noise=sn,
                batch_size=GRAVITY_BATCH
            )
            t_grav_rn = time.time() - t_rn

            rn_solves = 0
            rn_basins = 0
            rn_bprs = 0

            for ui_idx, orig_idx in enumerate(unsolved_idx):
                solved, new_model, new_unsat, n_basins, n_bpr, grav_only = \
                    process_instance_round(
                        helpers[orig_idx], engine_results[ui_idx],
                        inst_best_model[orig_idx], inst_best_unsat[orig_idx],
                        n_var, flips_rn
                    )
                rn_basins += n_basins
                rn_bprs += n_bpr

                if solved:
                    inst_solved[orig_idx] = True
                    inst_round_solved[orig_idx] = rnd
                    inst_best_model[orig_idx] = new_model
                    inst_best_unsat[orig_idx] = 0
                    rn_solves += 1
                    if grav_only:
                        inst_gravity_only[orig_idx] = True
                        total_grav_only += 1
                else:
                    if new_unsat < inst_best_unsat[orig_idx]:
                        inst_best_model[orig_idx] = new_model
                        inst_best_unsat[orig_idx] = new_unsat
                    elif new_model is not None:
                        inst_best_model[orig_idx] = new_model

            total_basins_found += rn_basins
            total_bpr_ran += rn_bprs
            per_round_solves.append(rn_solves)
            total_solved = sum(inst_solved)

            print(f'      → R{rnd}: +{rn_solves} solved ({total_solved}/{N_INST}), '
                  f'{rn_basins} basins, {rn_bprs} BPRs | '
                  f'grav={t_grav_rn:.0f}s, total={time.time()-t_rn:.0f}s')

            if total_solved == N_INST:
                break

        # ═══════════════════════════════════════════════════════════
        # RESULTS for this (α, n)
        # ═══════════════════════════════════════════════════════════
        elapsed = time.time() - t0
        solved_pct = sum(inst_solved) / N_INST * 100
        results[(alpha, n_var)] = solved_pct

        h15e = h15_ens[(alpha, n_var)]
        h19v = h19[(alpha, n_var)]
        h14e = h14_ens[(alpha, n_var)]
        h13e = h13_ens[(alpha, n_var)]
        delta = solved_pct - h15e

        rds = {
            'rounds_used': len(per_round_solves),
            'per_round': per_round_solves,
            'basins': total_basins_found,
            'bpr_ran': total_bpr_ran,
            'grav_only': total_grav_only,
            'round_solved': inst_round_solved[:],
            'unsolved_unsats': [inst_best_unsat[i] for i in range(N_INST)
                                if not inst_solved[i]],
        }
        round_stats[(alpha, n_var)] = rds

        tag = '★' if solved_pct >= 95 else ('▲' if delta > 2 else
              ('≈' if abs(delta) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {solved_pct:5.1f}% | '
              f'{total_grav_only:5d} | '
              f'{len(per_round_solves):4d} | '
              f'{total_basins_found:6d} | {total_bpr_ran:5d} | '
              f'{h15e:4.0f}% | {h19v:4.0f}% | {h14e:4.0f}% | {h13e:4.0f}% | '
              f'{delta:+6.1f}% | {elapsed:.0f}s {tag}')
    print('  ' + '-' * 120)


# ══════════════════════════════════════════════════════════════════════
#  FULL COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('FULL COMPARISON — H9b → H10b → H11b → H13 → H14 → H15 → H19 → H20')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | {'H11bE':>6} | "
      f"{'H13E':>5} | {'H14E':>5} | {'H15':>5} | {'H19':>5} | {'H20':>5} | {'ΔvsH15':>7}")
print('  ' + '-' * 100)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        h20v = results.get(key, 0)
        h15v = h15_ens[key]
        delta = h20v - h15v
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{h9b[key]:4.0f}% | {h10b[key]:4.0f}% | {h11b_ens[key]:5.1f}% | '
              f'{h13_ens[key]:4.0f}% | {h14_ens[key]:4.0f}% | '
              f'{h15v:4.0f}% | {h19[key]:4.0f}% | {h20v:4.0f}% | {delta:+6.1f}%')
    print('  ' + '-' * 100)


# ══════════════════════════════════════════════════════════════════════
#  ROUND ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 140)
print('ROUND ANALYSIS — which rounds solved instances?')
print('=' * 140)
print(f"  {'α':>5} | {'n':>5} | {'R1':>4} | {'R2':>4} | {'R3':>4} | "
      f"{'R4':>4} | {'R5':>4} | {'Fail':>4} | "
      f"{'Basins':>6} | {'BPRs':>5} | {'GravO':>5} | Interpretation")
print('  ' + '-' * 120)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats.get(key)
        if not rs:
            continue

        # Count per round
        round_counts = [0] * MAX_ROUNDS
        for rnd in rs['round_solved']:
            if rnd > 0:
                round_counts[rnd - 1] += 1
        n_fail = N_INST - sum(round_counts)

        parts = []
        if round_counts[0] > 0:
            parts.append(f'R1={round_counts[0]}')
        rescued = sum(round_counts[1:])
        if rescued > 0:
            parts.append(f'+{rescued} rescued')
        if n_fail > 0:
            parts.append(f'{n_fail} hard')
        interp = ', '.join(parts) if parts else 'All solved R1'

        rc = round_counts
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{rc[0]:4d} | {rc[1] if len(rc)>1 else 0:4d} | '
              f'{rc[2] if len(rc)>2 else 0:4d} | '
              f'{rc[3] if len(rc)>3 else 0:4d} | '
              f'{rc[4] if len(rc)>4 else 0:4d} | '
              f'{n_fail:4d} | '
              f'{rs["basins"]:6d} | {rs["bpr_ran"]:5d} | {rs["grav_only"]:5d} | '
              f'{interp}')
    print('  ' + '-' * 120)


# ══════════════════════════════════════════════════════════════════════
#  UNSOLVED INSTANCE ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('UNSOLVED INSTANCE ANALYSIS')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats.get(key)
        if not rs:
            continue
        unsats = rs['unsolved_unsats']
        if unsats:
            unsats_s = sorted(unsats)
            print(f'  α={alpha}, n={n_var}: {len(unsats)} unsolved | '
                  f'unsat: min={unsats_s[0]}, '
                  f'median={unsats_s[len(unsats_s)//2]}, '
                  f'max={unsats_s[-1]}, '
                  f'mean={np.mean(unsats_s):.1f}')


# ══════════════════════════════════════════════════════════════════════
#  BASIN EXPLORATION — UNSAT probability estimate
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('BASIN EXPLORATION — UNSAT probability estimate')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats.get(key)
        if not rs:
            continue
        basins = rs['basins']
        p = results.get(key, 0) / 100.0
        if basins > 0 and p < 1.0:
            p_miss = (1 - p) ** basins
            print(f'  α={alpha}, n={n_var}: basins={basins}, p̂={p:.2f}, '
                  f'P(miss)={(1-p):.3f}^{basins} = {p_miss:.2e}')


# ══════════════════════════════════════════════════════════════════════
#  COMPARISON CHART
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('H20 Deep Escalation + Basin Clustering — Solve Rate by n',
             fontsize=14, fontweight='bold')

all_versions = [
    ('H9b', h9b), ('H10b', h10b), ('H11bE', h11b_ens),
    ('H13E', h13_ens), ('H14E', h14_ens),
    ('H15', h15_ens), ('H19', h19),
    ('H20', results),
]
colors = ['#aaa', '#888', '#666', '#444', '#c80', '#08c', '#f44', '#0a0']

for ai, alpha in enumerate(ALPHAS):
    ax = axes[ai]
    for vi, (name, data) in enumerate(all_versions):
        vals = [data.get((alpha, n), 0) for n in NS]
        lw = 3 if name in ('H15', 'H19', 'H20') else 1.5
        ls = '-' if name in ('H15', 'H19', 'H20') else '--'
        marker = 'o' if name in ('H15', 'H19', 'H20') else '.'
        ax.plot(NS, vals, marker=marker, label=name, color=colors[vi],
                linewidth=lw, linestyle=ls)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('n (variables)')
    ax.set_ylabel('Solve Rate (%)')
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('h20_deep_wave_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: h20_deep_wave_results.png')

print(f'\nEffective budget per instance (max {MAX_ROUNDS} rounds):')
print(f'  R1 gravity: {STEPS_R1} steps × {P_R1} ptcl × {K_ENGINES} engines')
for i, esc in enumerate(ESCALATION):
    print(f'  R{i+2} gravity: {esc["steps"]} steps × {P_RN} ptcl × {K_ENGINES} engines '
          f'({int(esc["seed_frac"]*100)}% seed, σ={esc["noise"]:.1f})')
print(f'  BPR per basin: {N_BPR_PER_BASIN} attempts × FULL flip budget (no tiers)')
print(f'  Total compute: ~{K_ENGINES}× H15 gravity + {MAX_BASINS}×{N_BPR_PER_BASIN} BPR/round')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H21 — Structure-Aware Basin Search
# ══════════════════════════════════════════════════════════════════════
#
# THEORY — Why clause-signature clustering beats Hamming:
#
#   Hamming distance measures how DIFFERENT two assignments are.
#   But two candidates can be:
#     • Hamming-FAR yet violate the SAME hard clause cluster → same basin
#     • Hamming-CLOSE yet violate DIFFERENT clause clusters → different basins
#
#   Clause signatures measure WHETHER THEY FAIL THE SAME WAY.
#   This directly identifies basin identity in the energy landscape.
#
# ARCHITECTURE (changes from H20v3):
#   1. Clause-signature embedding: hash violated clauses into B=128
#      buckets, L2-normalize → S(x) ∈ ℝ^128
#   2. Basin clustering: cosine distance on S(x) instead of
#      Hamming distance on binary assignments
#   3. Basin memory: track BPR success rates by unsat-fraction
#      ranges across rounds (informational + adaptive allocation)
#
# EVERYTHING ELSE from H20v3 is preserved:
#   • K=8 engines × P=1000 particles
#   • Per-engine noise diversity in R2-R5
#   • FULL BPR budget (no tiers), N_BPR_PER_BASIN=2
#   • Clean gravity core (no anti-collapse/micro-reset)
#   • All GPU optimizations [A-F]
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════


# ═══════ H21 PARAMETERS ═══════
SIG_BUCKETS   = 128    # clause-signature dimension
SIG_TAU_COS   = 0.25   # cosine distance threshold (merge if dist < 0.25)
                        # equivalently: merge if cosine similarity > 0.75
N_BPR_BASE    = 2      # default BPR attempts per basin (= H20v3)
N_BPR_BOOST   = 3      # extra BPR for promising basins (memory says P>0.10)
N_BPR_MIN     = 1      # reduced BPR for unpromising (memory says P<0.02)
MEMORY_MIN_SAMPLES = 30 # min samples before memory influences allocation


# ══════════════════════════════════════════════════════════════════════
#  CLAUSE-SIGNATURE EMBEDDING (Numba JIT)
#
#  For each candidate assignment, compute which clauses are VIOLATED,
#  then hash each violated clause into B buckets using 2 hash functions.
#  The resulting vector tells you WHAT REGION of the formula is broken.
#
#  O(m) per candidate — essentially free.
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def clause_signature_numba(x_bin, clauses_v, clauses_s, n_buckets):
    """Compute clause-violation signature vector.

    Args:
        x_bin: (n,) int32 binary assignment (0/1)
        clauses_v: (m, 3) int32 variable indices
        clauses_s: (m, 3) int32 signs (+1/-1)
        n_buckets: int, signature dimension

    Returns:
        sig: (n_buckets,) float32, L2-normalized signature
    """
    m = clauses_v.shape[0]
    sig = np.zeros(n_buckets, dtype=np.float32)

    for c in range(m):
        # Check if clause c is satisfied
        sat = False
        for k in range(3):
            v = clauses_v[c, k]
            s = clauses_s[c, k]
            lit_val = (2 * x_bin[v] - 1) * s
            if lit_val > 0:
                sat = True
                break
        if not sat:
            # Violated clause → hash into buckets
            # Two independent hash functions for better distribution
            h1 = (c * 2654435761) % n_buckets   # Knuth multiplicative
            h2 = (c * 2246822519 + 1) % n_buckets  # second hash
            sig[h1] += 1.0
            sig[h2] += 1.0

    # L2 normalize
    norm_sq = np.float32(0.0)
    for b in range(n_buckets):
        norm_sq += sig[b] * sig[b]
    if norm_sq > 1e-16:
        inv_norm = np.float32(1.0) / np.sqrt(norm_sq)
        for b in range(n_buckets):
            sig[b] *= inv_norm

    return sig


# ══════════════════════════════════════════════════════════════════════
#  BASIN CLUSTERING BY CLAUSE SIGNATURES (Numba JIT)
#
#  Instead of Hamming distance on binary assignments, we cluster by
#  cosine distance on clause signatures. This correctly identifies
#  basins even when assignments are Hamming-distant but stuck on
#  the same clause cluster.
#
#  Complexity: O(K² × B) where K = candidates, B = SIG_BUCKETS
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def basin_cluster_signature_numba(signatures, unsat_counts,
                                  tau_cos, max_clusters):
    """Cluster candidates by clause-signature cosine distance.

    Args:
        signatures: (K, B) float32, L2-normalized clause signatures
        unsat_counts: (K,) int32, unsat count per candidate
        tau_cos: float, cosine distance threshold to merge (0.25 default)
        max_clusters: int, max basins to return

    Returns:
        centers: (max_clusters,) int32, indices of cluster centers
        n_clusters: int, number of clusters found
    """
    K = signatures.shape[0]
    B = signatures.shape[1]
    order = np.argsort(unsat_counts)  # best-first greedy
    centers = np.zeros(max_clusters, dtype=np.int32)
    n_clusters = 0

    for oi in range(K):
        i = order[oi]
        near_existing = False
        for c in range(n_clusters):
            ci = centers[c]
            # Cosine distance = 1 - dot(a, b) for L2-normalized vectors
            dot = np.float32(0.0)
            for b in range(B):
                dot += signatures[i, b] * signatures[ci, b]
            cos_dist = 1.0 - dot
            if cos_dist < tau_cos:
                near_existing = True
                break
        if not near_existing and n_clusters < max_clusters:
            centers[n_clusters] = i
            n_clusters += 1

    return centers, n_clusters


# ══════════════════════════════════════════════════════════════════════
#  BASIN MEMORY — Cross-Round BPR Statistics
#
#  Tracks BPR outcomes by unsat-fraction ranges across rounds.
#  Used for:
#   1. Adaptive BPR allocation (promising basins → more attempts)
#   2. End-of-run analysis (which difficulty ranges succeed?)
#
#  NOT cross-instance (different instances have different clause
#  structures, so raw signatures don't transfer).
# ══════════════════════════════════════════════════════════════════════

class BasinMemory:
    """Lightweight BPR outcome tracker by unsat-fraction bins."""

    def __init__(self, n_bins=20):
        self.n_bins = n_bins
        self.success = np.zeros(n_bins, dtype=np.float64)
        self.total = np.zeros(n_bins, dtype=np.float64)

    def _bin(self, unsat_frac):
        return min(self.n_bins - 1, int(unsat_frac * self.n_bins))

    def record(self, unsat_frac, did_succeed):
        b = self._bin(unsat_frac)
        self.total[b] += 1
        if did_succeed:
            self.success[b] += 1

    def success_rate(self, unsat_frac):
        """Estimate P(BPR success) for a given unsat fraction."""
        b = self._bin(unsat_frac)
        if self.total[b] < 3:
            return 0.5  # not enough data → neutral
        return self.success[b] / self.total[b]

    def total_samples(self):
        return int(self.total.sum())

    def bpr_attempts_for(self, unsat_frac):
        """Adaptive allocation: more attempts for promising basins."""
        if self.total_samples() < MEMORY_MIN_SAMPLES:
            return N_BPR_BASE  # cold start → default

        p = self.success_rate(unsat_frac)
        if p > 0.10:
            return N_BPR_BOOST  # promising → 3 attempts
        elif p < 0.02:
            return N_BPR_MIN    # unpromising → 1 attempt
        else:
            return N_BPR_BASE   # normal → 2 attempts

    def summary(self):
        """Print success rates by unsat-fraction bin."""
        lines = []
        for b in range(self.n_bins):
            if self.total[b] > 0:
                lo = b / self.n_bins * 100
                hi = (b + 1) / self.n_bins * 100
                rate = self.success[b] / self.total[b] * 100
                lines.append(
                    f'    unsat {lo:4.0f}%-{hi:4.0f}%: '
                    f'{int(self.success[b])}/{int(self.total[b])} '
                    f'({rate:5.1f}%)'
                )
        return '\n'.join(lines) if lines else '    (no data)'


# ══════════════════════════════════════════════════════════════════════
#  PROCESS ONE INSTANCE — H21 version
#
#  Key differences from H20v3:
#   1. Computes clause signatures for each candidate O(m)
#   2. Clusters by cosine distance on signatures (not Hamming)
#   3. Records BPR outcomes in basin memory
#   4. Adaptive BPR allocation via memory (after warm-up)
# ══════════════════════════════════════════════════════════════════════

def process_instance_round_h21(helper, candidates_list, best_model,
                               best_unsat, n, bpr_max_flips,
                               basin_memory):
    """Process one instance using clause-signature clustering.

    Returns: (solved, model, unsat, n_basins, n_bpr, grav_only)
    """
    m = helper.m
    candidates = candidates_list

    # Check gravity-only solves
    for c in candidates:
        if c['unsat'] == 0:
            return True, c['assignment'], 0, 0, 0, True

    # Stage 1: reject if unsat > 20% × n
    threshold = int(UNSAT_REJECT_FRAC * n)
    candidates = [c for c in candidates if c['unsat'] <= threshold]

    if not candidates:
        return False, best_model, best_unsat, 0, 0, False

    # Stage 2: sort by unsat, keep top
    candidates.sort(key=lambda c: c['unsat'])
    candidates = candidates[:TOP_CANDIDATES]

    # ───────────────────────────────────────────────────────────
    # H21 CHANGE: Compute clause signatures for each candidate
    # ───────────────────────────────────────────────────────────
    n_cand = len(candidates)
    signatures = np.zeros((n_cand, SIG_BUCKETS), dtype=np.float32)
    cand_unsat = np.array([c['unsat'] for c in candidates], dtype=np.int32)

    for i in range(n_cand):
        signatures[i] = clause_signature_numba(
            candidates[i]['assignment'],
            helper.clauses_v, helper.clauses_s,
            SIG_BUCKETS
        )

    # ───────────────────────────────────────────────────────────
    # H21 CHANGE: Cluster by clause-signature cosine distance
    # ───────────────────────────────────────────────────────────
    centers, n_clusters = basin_cluster_signature_numba(
        signatures, cand_unsat, SIG_TAU_COS, MAX_BASINS
    )
    basin_indices = [int(centers[c]) for c in range(n_clusters)]

    n_bpr_ran = 0
    solved = False
    new_best_model = best_model
    new_best_unsat = best_unsat

    if not basin_indices:
        return False, best_model, best_unsat, 0, 0, False

    # Submit BPR workers per basin with adaptive allocation
    futures = {}
    for bi in basin_indices:
        cand = candidates[bi]
        unsat_frac = cand['unsat'] / m

        # H21 CHANGE: adaptive BPR allocation via basin memory
        n_attempts = basin_memory.bpr_attempts_for(unsat_frac)

        for attempt in range(n_attempts):
            f = _BPR_POOL.submit(
                bpr_worker, helper.clauses_v, helper.clauses_s,
                cand['assignment'].copy(), cand['confidence'],
                bpr_max_flips, 0.5, 0.01, 0.1, BPR_BETA,
                CHAIN_PATIENCE, BRANCH_PATIENCE,
                COOL_RATE, STUB_FRAC, WEIGHT_BUMP, WEIGHT_DECAY
            )
            futures[f] = (bi, unsat_frac)
            n_bpr_ran += 1

    for f in as_completed(futures):
        bi, unsat_frac = futures[f]
        is_sat, flips_used, n_chains, sol = f.result()
        sol_unsat = 0 if is_sat else count_unsat(
            helper.clauses_v, helper.clauses_s, sol
        )

        # H21 CHANGE: record outcome in basin memory
        basin_memory.record(unsat_frac, is_sat)

        if is_sat:
            solved = True
            new_best_model = sol
            new_best_unsat = 0
            for remaining in futures:
                if remaining is not f:
                    remaining.cancel()
            break
        elif sol_unsat < new_best_unsat:
            new_best_unsat = sol_unsat
            new_best_model = sol.copy()

    return solved, new_best_model, new_best_unsat, n_clusters, n_bpr_ran, False


# ══════════════════════════════════════════════════════════════════════
#  COMPILE WARMUP — new Numba functions
# ══════════════════════════════════════════════════════════════════════

# Warmup clause_signature_numba
_wv21 = np.array([[0, 1, 2]], dtype=np.int32)
_ws21 = np.array([[1, -1, 1]], dtype=np.int32)
_wa21 = np.array([1, 0, 1], dtype=np.int32)
_ = clause_signature_numba(_wa21, _wv21, _ws21, SIG_BUCKETS)

# Warmup basin_cluster_signature_numba
_sigs = np.random.randn(4, SIG_BUCKETS).astype(np.float32)
_sigs /= np.linalg.norm(_sigs, axis=1, keepdims=True) + 1e-8
_unsats = np.array([5, 3, 8, 1], dtype=np.int32)
_ = basin_cluster_signature_numba(_sigs, _unsats, SIG_TAU_COS, 3)

print('✓ H21 — Structure-Aware Basin Search (clause-signature clustering)')
print()
print('  UPGRADES over H20v3:')
print(f'    1. Clause-signature clustering: B={SIG_BUCKETS} buckets, '
      f'τ_cos={SIG_TAU_COS} (cosine dist)')
print(f'       → clusters by WHICH CLAUSES are violated, not Hamming on assignments')
print(f'    2. Basin memory: adaptive BPR allocation '
      f'({N_BPR_BOOST} boost / {N_BPR_BASE} base / {N_BPR_MIN} min)')
print(f'       → learns P(success|unsat_frac) across rounds, '
      f'active after {MEMORY_MIN_SAMPLES} samples')
print()
print('  PRESERVED from H20v3:')
print(f'    • K={K_ENGINES} engines × {P_R1} ptcl/engine')
print(f'    • Per-engine noise diversity 1.0×–3.0× in R2-R5')
print(f'    • FULL BPR flip budget (no tiers)')
print(f'    • Clean gravity core (no anti-collapse/micro-reset)')
print(f'    • All GPU optimizations [A-F]')
print()
print('  WHY clause signatures beat Hamming:')
print('    Hamming: "how different are assignments A and B?"')
print('    Clause-sig: "do A and B violate the SAME clauses?"')
print('    → A and B can be Hamming-far but stuck on same hard clauses = same basin')
print('    → A and B can be Hamming-close but violating different clauses = different basins')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H21 Experiment: Structure-Aware Basin Search
# ══════════════════════════════════════════════════════════════════════

# ----- Baselines (same as Cell 2) -----
h20_baseline = {}  # will be populated from Cell 2 results if available

h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
h13_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 94.0,  (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 30.0,  (4.2, 750): 32.0,  (4.2, 1000): 16.0,
}
h14_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 42.0,  (4.2, 750): 28.0,  (4.2, 1000): 20.0,
}
h15_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 100.0,
    (4.2, 500): 54.0,  (4.2, 750): 36.0,  (4.2, 1000): 24.0,
}
h19 = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 92.0,  (4.0, 750): 96.0,  (4.0, 1000): 80.0,
    (4.2, 500): 22.0,  (4.2, 750): 14.0,  (4.2, 1000):  2.0,
}

ALPHAS   = [4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50

results_h21 = {}
round_stats_h21 = {}
all_basin_memories = {}  # store per (α,n) for analysis

print('=' * 150)
print('H21 — Structure-Aware Basin Search (clause-signature clustering)')
print(f'  K={K_ENGINES} engines | R1: {STEPS_R1} steps × {P_R1} ptcl | '
      f'R2-R5: escalating seed+noise × {P_RN} ptcl')
print(f'  Signature: B={SIG_BUCKETS} | τ_cos={SIG_TAU_COS} | Max {MAX_BASINS} basins | '
      f'BPR: {N_BPR_BASE} base / {N_BPR_BOOST} boost / {N_BPR_MIN} min (adaptive)')
print('=' * 150)
print(f"  {'α':>5} | {'n':>5} | {'Solved':>6} | {'GravO':>5} | "
      f"{'Rnds':>4} | {'Basins':>6} | {'BPRs':>5} | "
      f"{'H15':>5} | {'H19':>5} | {'H14':>5} | {'H13':>5} | "
      f"{'ΔvsH15':>7} | Time")
print('  ' + '-' * 130)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        # Basin memory for this (α, n) run — shared across all 50 instances
        basin_mem = BasinMemory(n_bins=20)

        # ═══════════════════════════════════════════════════════════
        # Generate all instances
        # ═══════════════════════════════════════════════════════════
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        helpers = [InstanceHelper(n_var, all_instances[i]) for i in range(N_INST)]

        # ═══════════════════════════════════════════════════════════
        # TRACKING
        # ═══════════════════════════════════════════════════════════
        inst_solved = [False] * N_INST
        inst_round_solved = [0] * N_INST
        inst_best_model = [None] * N_INST
        inst_best_unsat = [m_cls] * N_INST
        inst_gravity_only = [False] * N_INST

        total_grav_only = 0
        total_basins_found = 0
        total_bpr_ran = 0
        per_round_solves = []

        # ═══════════════════════════════════════════════════════════
        # ROUND 1: K engines × deep gravity (random+diverse seeds)
        # ═══════════════════════════════════════════════════════════
        t_r1 = time.time()
        print(f'    α={alpha}, n={n_var}: R1 — {N_INST} instances, '
              f'K={K_ENGINES} engines × {P_R1} ptcl × {STEPS_R1} steps')

        seeds_list = [generate_diverse_seeds(K_ENGINES, n_var, None)
                      for _ in range(N_INST)]

        engine_results = run_round_gravity(
            helpers, seeds_list, n_var, m_cls,
            particles=P_R1, steps=STEPS_R1,
            batch_size=GRAVITY_BATCH
        )
        t_grav_r1 = time.time() - t_r1

        r1_solves = 0
        r1_basins = 0
        r1_bprs = 0
        for inst in range(N_INST):
            solved, new_model, new_unsat, n_basins, n_bpr, grav_only = \
                process_instance_round_h21(
                    helpers[inst], engine_results[inst],
                    inst_best_model[inst], inst_best_unsat[inst],
                    n_var, FLIPS_R1, basin_mem
                )
            r1_basins += n_basins
            r1_bprs += n_bpr

            if solved:
                inst_solved[inst] = True
                inst_round_solved[inst] = 1
                inst_best_model[inst] = new_model
                inst_best_unsat[inst] = 0
                r1_solves += 1
                if grav_only:
                    inst_gravity_only[inst] = True
                    total_grav_only += 1
            else:
                inst_best_model[inst] = new_model if new_model is not None else inst_best_model[inst]
                inst_best_unsat[inst] = new_unsat

        total_basins_found += r1_basins
        total_bpr_ran += r1_bprs
        per_round_solves.append(r1_solves)
        total_solved = sum(inst_solved)

        print(f'      → R1: +{r1_solves} solved ({total_solved}/{N_INST}), '
              f'{r1_basins} basins, {r1_bprs} BPRs | '
              f'grav={t_grav_r1:.0f}s, total={time.time()-t_r1:.0f}s | '
              f'memory: {basin_mem.total_samples()} samples')

        # ═══════════════════════════════════════════════════════════
        # ROUNDS 2-5: Escalating re-ignition
        # ═══════════════════════════════════════════════════════════
        for rnd_idx, esc in enumerate(ESCALATION):
            rnd = rnd_idx + 2

            unsolved_idx = [i for i in range(N_INST) if not inst_solved[i]]
            if not unsolved_idx:
                per_round_solves.append(0)
                continue

            sf = esc['seed_frac']
            sn = esc['noise']
            steps_rn = esc['steps']
            flips_rn = esc['flips']

            t_rn = time.time()
            print(f'    α={alpha}, n={n_var}: R{rnd} — {len(unsolved_idx)} unsolved, '
                  f'K={K_ENGINES} engines × {P_RN} ptcl × {steps_rn} steps '
                  f'({int(sf*100)}% seed, σ={sn:.1f})')

            unsolved_helpers = [helpers[i] for i in unsolved_idx]
            global_bests = [inst_best_model[i] for i in unsolved_idx]

            engine_results = run_round_seeded_gravity(
                unsolved_helpers, global_bests, n_var, m_cls,
                particles=P_RN, steps=steps_rn,
                seed_frac=sf, seed_noise=sn,
                batch_size=GRAVITY_BATCH
            )
            t_grav_rn = time.time() - t_rn

            rn_solves = 0
            rn_basins = 0
            rn_bprs = 0

            for ui_idx, orig_idx in enumerate(unsolved_idx):
                solved, new_model, new_unsat, n_basins, n_bpr, grav_only = \
                    process_instance_round_h21(
                        helpers[orig_idx], engine_results[ui_idx],
                        inst_best_model[orig_idx], inst_best_unsat[orig_idx],
                        n_var, flips_rn, basin_mem
                    )
                rn_basins += n_basins
                rn_bprs += n_bpr

                if solved:
                    inst_solved[orig_idx] = True
                    inst_round_solved[orig_idx] = rnd
                    inst_best_model[orig_idx] = new_model
                    inst_best_unsat[orig_idx] = 0
                    rn_solves += 1
                    if grav_only:
                        inst_gravity_only[orig_idx] = True
                        total_grav_only += 1
                else:
                    if new_unsat < inst_best_unsat[orig_idx]:
                        inst_best_model[orig_idx] = new_model
                        inst_best_unsat[orig_idx] = new_unsat
                    elif new_model is not None:
                        inst_best_model[orig_idx] = new_model

            total_basins_found += rn_basins
            total_bpr_ran += rn_bprs
            per_round_solves.append(rn_solves)
            total_solved = sum(inst_solved)

            print(f'      → R{rnd}: +{rn_solves} solved ({total_solved}/{N_INST}), '
                  f'{rn_basins} basins, {rn_bprs} BPRs | '
                  f'grav={t_grav_rn:.0f}s, total={time.time()-t_rn:.0f}s | '
                  f'memory: {basin_mem.total_samples()} samples')

            if total_solved == N_INST:
                break

        # ═══════════════════════════════════════════════════════════
        # RESULTS for this (α, n)
        # ═══════════════════════════════════════════════════════════
        elapsed = time.time() - t0
        solved_pct = sum(inst_solved) / N_INST * 100
        results_h21[(alpha, n_var)] = solved_pct

        h15e = h15_ens[(alpha, n_var)]
        h19v = h19[(alpha, n_var)]
        h14e = h14_ens[(alpha, n_var)]
        h13e = h13_ens[(alpha, n_var)]
        delta = solved_pct - h15e

        rds = {
            'rounds_used': len(per_round_solves),
            'per_round': per_round_solves,
            'basins': total_basins_found,
            'bpr_ran': total_bpr_ran,
            'grav_only': total_grav_only,
            'round_solved': inst_round_solved[:],
            'unsolved_unsats': [inst_best_unsat[i] for i in range(N_INST)
                                if not inst_solved[i]],
        }
        round_stats_h21[(alpha, n_var)] = rds

        # Store basin memory for post-run analysis
        all_basin_memories[(alpha, n_var)] = basin_mem

        tag = '★' if solved_pct >= 95 else ('▲' if delta > 2 else
              ('≈' if abs(delta) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {solved_pct:5.1f}% | '
              f'{total_grav_only:5d} | '
              f'{len(per_round_solves):4d} | '
              f'{total_basins_found:6d} | {total_bpr_ran:5d} | '
              f'{h15e:4.0f}% | {h19v:4.0f}% | {h14e:4.0f}% | {h13e:4.0f}% | '
              f'{delta:+6.1f}% | {elapsed:.0f}s {tag}')
    print('  ' + '-' * 130)


# ══════════════════════════════════════════════════════════════════════
#  FULL COMPARISON TABLE — H21 vs all
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 150)
print('FULL COMPARISON — H9b → H13 → H14 → H15 → H19 → H21')
print('=' * 150)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H13E':>5} | "
      f"{'H14E':>5} | {'H15':>5} | {'H19':>5} | {'H21':>5} | "
      f"{'ΔvsH15':>7} | {'ΔvsH19':>7}")
print('  ' + '-' * 100)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        h21v = results_h21.get(key, 0)
        h15v = h15_ens[key]
        h19v = h19[key]
        delta15 = h21v - h15v
        delta19 = h21v - h19v
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{h9b[key]:4.0f}% | {h13_ens[key]:4.0f}% | '
              f'{h14_ens[key]:4.0f}% | '
              f'{h15v:4.0f}% | {h19v:4.0f}% | {h21v:4.0f}% | '
              f'{delta15:+6.1f}% | {delta19:+6.1f}%')
    print('  ' + '-' * 100)


# ══════════════════════════════════════════════════════════════════════
#  ROUND ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 150)
print('ROUND ANALYSIS — which rounds solved instances?')
print('=' * 150)
print(f"  {'α':>5} | {'n':>5} | {'R1':>4} | {'R2':>4} | {'R3':>4} | "
      f"{'R4':>4} | {'R5':>4} | {'Fail':>4} | "
      f"{'Basins':>6} | {'BPRs':>5} | {'GravO':>5} | Interpretation")
print('  ' + '-' * 130)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats_h21.get(key)
        if not rs:
            continue

        round_counts = [0] * MAX_ROUNDS
        for rnd in rs['round_solved']:
            if rnd > 0:
                round_counts[rnd - 1] += 1
        n_fail = N_INST - sum(round_counts)

        parts = []
        if round_counts[0] > 0:
            parts.append(f'R1={round_counts[0]}')
        rescued = sum(round_counts[1:])
        if rescued > 0:
            parts.append(f'+{rescued} rescued')
        if n_fail > 0:
            parts.append(f'{n_fail} hard')
        interp = ', '.join(parts) if parts else 'All solved R1'

        rc = round_counts
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{rc[0]:4d} | {rc[1] if len(rc)>1 else 0:4d} | '
              f'{rc[2] if len(rc)>2 else 0:4d} | '
              f'{rc[3] if len(rc)>3 else 0:4d} | '
              f'{rc[4] if len(rc)>4 else 0:4d} | '
              f'{n_fail:4d} | '
              f'{rs["basins"]:6d} | {rs["bpr_ran"]:5d} | {rs["grav_only"]:5d} | '
              f'{interp}')
    print('  ' + '-' * 130)


# ══════════════════════════════════════════════════════════════════════
#  UNSOLVED INSTANCE ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('UNSOLVED INSTANCE ANALYSIS (H21)')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats_h21.get(key)
        if not rs:
            continue
        unsats = rs['unsolved_unsats']
        if unsats:
            unsats_s = sorted(unsats)
            print(f'  α={alpha}, n={n_var}: {len(unsats)} unsolved | '
                  f'unsat: min={unsats_s[0]}, '
                  f'median={unsats_s[len(unsats_s)//2]}, '
                  f'max={unsats_s[-1]}, '
                  f'mean={np.mean(unsats_s):.1f}')


# ══════════════════════════════════════════════════════════════════════
#  BASIN MEMORY ANALYSIS — P(success) by unsat-fraction
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('BASIN MEMORY ANALYSIS — BPR success rate by unsat-fraction')
print('  (cross-round learning within each (α,n) run)')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        mem = all_basin_memories.get(key)
        if mem:
            print(f'\n  α={alpha}, n={n_var}: '
                  f'{mem.total_samples()} total BPR attempts')
            print(mem.summary())


# ══════════════════════════════════════════════════════════════════════
#  COMPARISON CHART
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating H21 charts...')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H21 Structure-Aware Basin Search — Solve Rate by n',
             fontsize=14, fontweight='bold')

all_versions = [
    ('H9b', h9b), ('H13E', h13_ens), ('H14E', h14_ens),
    ('H15', h15_ens), ('H19', h19), ('H21', results_h21),
]
colors = ['#aaa', '#444', '#c80', '#08c', '#f44', '#0a0']

for ai, alpha in enumerate(ALPHAS):
    ax = axes[ai]
    for vi, (name, data) in enumerate(all_versions):
        vals = [data.get((alpha, n), 0) for n in NS]
        lw = 3 if name in ('H15', 'H21') else 1.5
        ls = '-' if name in ('H15', 'H19', 'H21') else '--'
        marker = 'o' if name in ('H15', 'H19', 'H21') else '.'
        ax.plot(NS, vals, marker=marker, label=name, color=colors[vi],
                linewidth=lw, linestyle=ls)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('n (variables)')
    ax.set_ylabel('Solve Rate (%)')
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('h21_clause_sig_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: h21_clause_sig_results.png')

print(f'\nH21 configuration:')
print(f'  Clause signatures: B={SIG_BUCKETS} buckets, '
      f'2 hash functions, L2-normalized')
print(f'  Clustering: cosine distance < {SIG_TAU_COS} to merge, '
      f'max {MAX_BASINS} basins')
print(f'  BPR allocation: {N_BPR_BOOST} (promising) / '
      f'{N_BPR_BASE} (normal) / {N_BPR_MIN} (unpromising)')
print(f'  Memory: active after {MEMORY_MIN_SAMPLES} BPR samples')
print(f'  All other params: identical to H20v3')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H22 — Ultimate Basin Search
# ══════════════════════════════════════════════════════════════════════
#
# FOUR NEW SYSTEMS layered on top of H21:
#
#   1. CONFLICT-CORE MICRO-SOLVER  (Numba)
#      When BPR returns with 1-5 unsat clauses, extract the 3-15
#      involved variables and exhaustively try all 2^k assignments.
#      O(2^15) = 32K operations — trivial cost, converts near-misses.
#
#   2. CONFLICT-DEGREE WEIGHTED BPR  (Numba)
#      Modified WalkSAT: greedy variable selection now adds a bonus
#      for variables appearing in MULTIPLE violated clauses. Flipping
#      a "conflict hub" fixes several clauses at once.
#
#   3. COMMUNITY-AWARE SEEDING  (Numba)
#      Build Variable Interaction Graph (VIG) — edge between variables
#      sharing a clause. Detect communities via label propagation.
#      Generate seeds by flipping whole communities, not random bits.
#      Produces structurally diverse starting points for gravity.
#
#   4. BELIEF PROPAGATION WARM-START  (Triton GPU kernel + fallback)
#      Run BP on the factor graph to compute per-variable marginals
#      P(x_i = 1). Use marginals to initialize gravity particles near
#      the BP-suggested solution. High-confidence vars (P > 0.95) are
#      pinned. This dramatically narrows the search space.
#
# PRESERVED from H21:
#   • Clause-signature clustering, basin memory
#   • K=8 engines × P=1000, per-engine noise diversity
#   • FULL BPR budget, clean gravity core, all GPU opts [A-F]
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════

# ─── Triton availability ───
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False

print(f'Triton available: {HAS_TRITON}')


# ══════════════════════════════════════════════════════════════════════
#  1. CONFLICT-CORE MICRO-SOLVER  (Numba JIT)
#
#  When BPR stalls at 1-5 unsatisfied clauses, the "core" variables
#  (3-15 vars in those clauses) are locked — every flip breaks another
#  clause. Brute-forcing 2^k assignments on just those vars costs
#  at most 2^15 × m = ~130M ops. At α=4.2 this converts the
#  near-misses that BPR repeatedly reaches but cannot close.
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def conflict_core_solver(clauses_v, clauses_s, assignment,
                         max_core_vars=15, max_unsat_clauses=5):
    """Exhaustive search on the conflict core.

    Args:
        clauses_v: (m, 3) int32 variable indices
        clauses_s: (m, 3) int32 signs (+1/-1)
        assignment: (n,) int32 binary assignment
        max_core_vars: int, abort if core larger than this
        max_unsat_clauses: int, only attempt if ≤ this many violated

    Returns:
        solved: bool
        best_assignment: (n,) int32
        best_unsat: int
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]

    # Find violated clauses and mark core variables
    var_in_core = np.zeros(n, dtype=np.int8)
    n_violated = 0

    for c in range(m):
        sat = False
        for k in range(3):
            v = clauses_v[c, k]
            s = clauses_s[c, k]
            if (assignment[v] == 1 and s == 1) or (assignment[v] == 0 and s == -1):
                sat = True
                break
        if not sat:
            n_violated += 1
            if n_violated > max_unsat_clauses:
                return False, assignment, n_violated
            for k in range(3):
                var_in_core[clauses_v[c, k]] = 1

    if n_violated == 0:
        return True, assignment, 0

    # Collect core variables
    core_vars = np.empty(3 * max_unsat_clauses, dtype=np.int32)
    n_core = 0
    for i in range(n):
        if var_in_core[i] == 1:
            core_vars[n_core] = i
            n_core += 1
            if n_core > max_core_vars:
                return False, assignment, n_violated

    # Brute-force all 2^k assignments on core variables
    best = assignment.copy()
    best_unsat = n_violated

    for bits in range(1 << n_core):
        # Apply core assignment
        trial = assignment.copy()
        for i in range(n_core):
            trial[core_vars[i]] = (bits >> i) & 1

        # Count unsatisfied clauses
        unsat = 0
        for c in range(m):
            sat = False
            for k in range(3):
                v = clauses_v[c, k]
                s = clauses_s[c, k]
                if (trial[v] == 1 and s == 1) or (trial[v] == 0 and s == -1):
                    sat = True
                    break
            if not sat:
                unsat += 1
                if unsat >= best_unsat:
                    break  # prune — can't beat current best

        if unsat == 0:
            return True, trial, 0
        if unsat < best_unsat:
            best_unsat = unsat
            for i in range(n):
                best[i] = trial[i]

    return best_unsat == 0, best, best_unsat


# ══════════════════════════════════════════════════════════════════════
#  2. CONFLICT-DEGREE WEIGHTED BPR  (Numba JIT)
#
#  Modified WalkSAT with one key change:
#  In the greedy variable selection step, the score now includes a
#  "cross-clause bonus" — variables appearing in MULTIPLE currently
#  violated clauses get a multiplicative boost. This encourages
#  flipping "hub" variables that fix several clauses at once.
#
#  The full BPR chain (tree-walk, chain death, stubbornness) is
#  preserved; only the scoring formula in the greedy step changes.
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_chain_weighted(clauses_v, clauses_s, assignment, weight,
                       max_flips=200000, T_init=0.5, T_min=0.01,
                       p_random=0.1, beta=0.3,
                       chain_patience=5000, branch_patience=80,
                       cool_rate=0.95, stub_frac=0.08,
                       weight_bump=2.0, weight_decay=0.9,
                       cross_clause_bonus=0.15):
    """BPR with conflict-degree weighted variable selection.

    Args:
        cross_clause_bonus: float, multiplicative bonus per additional
            violated clause containing candidate variable (default 0.15)
    All other args: identical to bpr_chain.
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]
    n_stub = max(2, int(stub_frac * n))

    # Build var→clause adjacency
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for vi in range(n):
        var_off[vi + 1] = var_off[vi] + var_count[vi]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            pos = var_off[vi] + fill[vi]
            var_adj[pos] = c
            var_sign[pos] = clauses_s[c, j]
            fill[vi] += 1

    clause_w = np.ones(m, dtype=np.float64)
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            vi = clauses_v[c, j]
            s_ = clauses_s[c, j]
            if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                clause_sat[c] += 1

    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    best_n_unsat = n_unsat
    best_assign = assignment.copy()
    chain_id = 0
    chain_stale = 0
    chain_best_unsat = n_unsat
    in_branch = False
    branch_stale = 0
    recent_size = 8
    recent_flipped = np.full(recent_size, -1, dtype=np.int32)
    recent_idx = 0
    stubbornness = np.zeros(n, dtype=np.float64)

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip, chain_id + 1, 0

        if n_unsat < best_n_unsat:
            best_n_unsat = n_unsat
            for i in range(n):
                best_assign[i] = assignment[i]
            chain_stale = 0
            chain_best_unsat = n_unsat
        elif n_unsat < chain_best_unsat:
            chain_best_unsat = n_unsat
            chain_stale = 0
        else:
            chain_stale += 1

        # Chain death — restart from best + stubborn flips
        if chain_stale >= chain_patience and n_unsat > 0:
            for ui in range(n_unsat):
                clause_w[unsat_list[ui]] += weight_bump
            for c in range(m):
                clause_w[c] *= weight_decay
            for vi in range(n):
                stubbornness[vi] = 0.0
            for ui in range(n_unsat):
                cc = unsat_list[ui]
                w_c = clause_w[cc]
                for j in range(3):
                    stubbornness[clauses_v[cc, j]] += w_c
            stub_to_flip = np.zeros(n_stub, dtype=np.int32)
            stub_used = np.zeros(n, dtype=np.int8)
            for k in range(n_stub):
                best_sv = -1.0
                best_vi = 0
                for vi in range(n):
                    if stub_used[vi] == 0 and stubbornness[vi] > best_sv:
                        best_sv = stubbornness[vi]
                        best_vi = vi
                stub_to_flip[k] = best_vi
                stub_used[best_vi] = 1
            for i in range(n):
                assignment[i] = best_assign[i]
            for k in range(n_stub):
                assignment[stub_to_flip[k]] = 1 - assignment[stub_to_flip[k]]
            n_random = max(1, n // 100)
            for _ in range(n_random):
                vi = np.random.randint(n)
                if np.random.random() < 0.3:
                    assignment[vi] = 1 - assignment[vi]

            n_unsat = 0
            for c in range(m):
                clause_sat[c] = 0
                for j in range(3):
                    vi = clauses_v[c, j]
                    s_ = clauses_s[c, j]
                    if (assignment[vi] == 1 and s_ == 1) or (assignment[vi] == 0 and s_ == -1):
                        clause_sat[c] += 1
                if clause_sat[c] == 0:
                    unsat_pos[c] = n_unsat
                    unsat_list[n_unsat] = c
                    n_unsat += 1
                else:
                    unsat_pos[c] = -1

            if n_unsat == 0:
                return assignment, flip, chain_id + 1, 0
            chain_id += 1
            chain_stale = 0
            chain_best_unsat = n_unsat
            in_branch = False
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0
            continue

        # Branch focusing — follow weighted violated neighbors
        ci = -1
        if in_branch and branch_stale < branch_patience:
            best_neighbor_w = -1.0
            for ri in range(recent_size):
                rv = recent_flipped[ri]
                if rv < 0:
                    continue
                for idx in range(var_off[rv], var_off[rv + 1]):
                    cc = var_adj[idx]
                    if clause_sat[cc] == 0 and clause_w[cc] > best_neighbor_w:
                        best_neighbor_w = clause_w[cc]
                        ci = cc
            if ci < 0:
                in_branch = False

        if not in_branch or ci < 0:
            ci = unsat_list[np.random.randint(n_unsat)]
            in_branch = True
            branch_stale = 0
            for ri in range(recent_size):
                recent_flipped[ri] = -1
            recent_idx = 0

        # Temperature schedule
        T_max_chain = T_init * (cool_rate ** min(chain_id, 30))
        local_prog = min(1.0, chain_stale / chain_patience)
        T = T_min + (T_max_chain - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * local_prog))

        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            # ─── H22 CHANGE: conflict-degree weighted scoring ───
            int_brks = np.zeros(3, dtype=np.int32)
            scores = np.zeros(3, dtype=np.float64)
            for j in range(3):
                v_cand = clauses_v[ci, j]
                i_brk = 0
                w_brk = 0.0
                w_make = 0.0
                n_cross = 0  # NEW: count OTHER violated clauses with this var
                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc2 = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies = ((assignment[v_cand] == 1 and s_here == 1) or
                                 (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies:
                        if clause_sat[cc2] == 1:
                            i_brk += 1
                            w_brk += clause_w[cc2]
                    else:
                        if clause_sat[cc2] == 0:
                            w_make += clause_w[cc2]
                            if cc2 != ci:
                                n_cross += 1  # other violated clause
                int_brks[j] = i_brk
                delta = w_brk - w_make
                # H22: boost score for multi-clause conflict hubs
                cross_mult = 1.0 + cross_clause_bonus * n_cross
                scores[j] = (np.exp(-delta / (T + 1e-10))
                             * (1.0 + beta * weight[v_cand])
                             * cross_mult)

            zb_n = 0
            zb_opts = np.zeros(3, dtype=np.int32)
            for j in range(3):
                if int_brks[j] == 0:
                    zb_opts[zb_n] = j
                    zb_n += 1
            if zb_n > 0:
                v_flip = clauses_v[ci, zb_opts[np.random.randint(zb_n)]]
            else:
                total = scores[0] + scores[1] + scores[2]
                if total < 1e-30:
                    v_flip = clauses_v[ci, np.random.randint(3)]
                else:
                    r = np.random.random() * total
                    if r <= scores[0]:
                        v_flip = clauses_v[ci, 0]
                    elif r <= scores[0] + scores[1]:
                        v_flip = clauses_v[ci, 1]
                    else:
                        v_flip = clauses_v[ci, 2]

        # Apply flip + update clause_sat / unsat tracking
        old_n_unsat = n_unsat
        assignment[v_flip] = 1 - assignment[v_flip]
        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc2 = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc2]
            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc2] += 1
            else:
                clause_sat[cc2] -= 1
            new_sat = clause_sat[cc2]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc2]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc2] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc2
                unsat_pos[cc2] = n_unsat
                n_unsat += 1

        if n_unsat < old_n_unsat:
            branch_stale = 0
        else:
            branch_stale += 1
        recent_flipped[recent_idx % recent_size] = v_flip
        recent_idx += 1

    return best_assign, max_flips, chain_id + 1, best_n_unsat


# ══════════════════════════════════════════════════════════════════════
#  WEIGHTED BPR WORKER (drop-in for bpr_worker)
# ══════════════════════════════════════════════════════════════════════

def bpr_worker_weighted(clauses_v, clauses_s, x_np, weight,
                        max_flips, T_init, T_min, p_random, beta,
                        chain_patience, branch_patience,
                        cool_rate, stub_frac, weight_bump, weight_decay):
    sol, flips, n_chains, remaining = bpr_chain_weighted(
        clauses_v, clauses_s, x_np, weight,
        max_flips=max_flips, T_init=T_init, T_min=T_min,
        p_random=p_random, beta=beta,
        chain_patience=chain_patience, branch_patience=branch_patience,
        cool_rate=cool_rate, stub_frac=stub_frac,
        weight_bump=weight_bump, weight_decay=weight_decay
    )
    return (remaining == 0), flips, n_chains, sol


# ══════════════════════════════════════════════════════════════════════
#  3. COMMUNITY-AWARE SEEDING  (Numba JIT)
#
#  Build the Variable Interaction Graph (VIG):
#    - Node per variable
#    - Edge between variables sharing at least one clause
#  Detect communities via label propagation (O(m) per iteration).
#  Generate seeds by flipping whole communities, producing
#  structurally diverse starting points instead of random noise.
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def build_vig_communities(clauses_v, n, max_iter=15):
    """Build VIG and detect communities via label propagation.

    Args:
        clauses_v: (m, 3) int32 variable indices
        n: int, number of variables
        max_iter: int, label propagation iterations

    Returns:
        labels: (n,) int32, community label for each variable
        n_communities: int, number of unique communities
    """
    m = clauses_v.shape[0]

    # Build adjacency list — each clause creates 3 edges: (v0,v1),(v0,v2),(v1,v2)
    # Count degrees first
    degree = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            degree[clauses_v[c, j]] += 2  # each clause adds 2 edges per var

    # CSR offsets
    adj_off = np.zeros(n + 1, dtype=np.int32)
    for i in range(n):
        adj_off[i + 1] = adj_off[i] + degree[i]
    adj = np.empty(adj_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)

    for c in range(m):
        v0, v1, v2 = clauses_v[c, 0], clauses_v[c, 1], clauses_v[c, 2]
        # v0-v1
        adj[adj_off[v0] + fill[v0]] = v1; fill[v0] += 1
        adj[adj_off[v1] + fill[v1]] = v0; fill[v1] += 1
        # v0-v2
        adj[adj_off[v0] + fill[v0]] = v2; fill[v0] += 1
        adj[adj_off[v2] + fill[v2]] = v0; fill[v2] += 1
        # v1-v2
        adj[adj_off[v1] + fill[v1]] = v2; fill[v1] += 1
        adj[adj_off[v2] + fill[v2]] = v1; fill[v2] += 1

    # Label propagation
    labels = np.arange(n, dtype=np.int32)  # each node starts as own label

    for iteration in range(max_iter):
        changed = False
        # Process nodes in random order
        order = np.random.permutation(n).astype(np.int32)
        for oi in range(n):
            node = order[oi]
            if adj_off[node + 1] == adj_off[node]:
                continue

            # Count neighbor labels
            label_count = np.zeros(n, dtype=np.int32)
            best_label = labels[node]
            best_count = 0
            for idx in range(adj_off[node], adj_off[node + 1]):
                nb = adj[idx]
                lb = labels[nb]
                label_count[lb] += 1
                if label_count[lb] > best_count:
                    best_count = label_count[lb]
                    best_label = lb

            if best_label != labels[node]:
                labels[node] = best_label
                changed = True

        if not changed:
            break

    # Count unique communities
    seen = np.zeros(n, dtype=np.int8)
    n_communities = 0
    for i in range(n):
        if seen[labels[i]] == 0:
            seen[labels[i]] = 1
            n_communities += 1

    return labels, n_communities


def generate_community_seeds(K, n, best_model, community_labels,
                             n_communities):
    """Generate K seeds with community-aware diversity.

    Mix of strategies:
      - community-flip: flip all variables in 1-3 random communities
      - community-targeted: flip the community with most conflict overlap
      - standard diverse: random/complement/noise (fallback)
    """
    seeds = np.zeros((K, n), dtype=np.int32)

    # Collect community members
    communities = {}
    for i in range(n):
        lb = int(community_labels[i])
        if lb not in communities:
            communities[lb] = []
        communities[lb].append(i)
    comm_ids = list(communities.keys())

    for k in range(K):
        if best_model is None:
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)
            continue

        r = np.random.random()
        if r < 0.35 and len(comm_ids) >= 2:
            # Flip 1-3 random communities
            seeds[k] = best_model.copy()
            n_to_flip = min(len(comm_ids), np.random.randint(1, 4))
            chosen = np.random.choice(len(comm_ids), n_to_flip, replace=False)
            for ci in chosen:
                for vi in communities[comm_ids[ci]]:
                    seeds[k][vi] = 1 - seeds[k][vi]
        elif r < 0.55:
            # Complement of best
            seeds[k] = 1 - best_model
        elif r < 0.75:
            # Small noise (~5% flip)
            seeds[k] = best_model.copy()
            nf = max(1, int(0.05 * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        elif r < 0.90:
            # Moderate noise (~20% flip)
            seeds[k] = best_model.copy()
            frac = 0.15 + 0.10 * np.random.random()
            nf = max(1, int(frac * n))
            idx = np.random.choice(n, nf, replace=False)
            seeds[k][idx] = 1 - seeds[k][idx]
        else:
            # Full random
            seeds[k] = np.random.randint(0, 2, n).astype(np.int32)

    return seeds


# ══════════════════════════════════════════════════════════════════════
#  4. BELIEF PROPAGATION  (Triton GPU kernel + Numba fallback)
#
#  BP on the 3-SAT factor graph computes marginal probabilities
#  P(x_i = 1) for each variable. These marginals warm-start gravity
#  particles near the likely solution.
#
#  Factor graph: bipartite (n variables ↔ m clauses), 3m edges.
#  Messages: for each edge (clause a, variable i):
#    warning[e] = P(clause a NEEDS variable i)
#              = ∏_{j∈clause_a, j≠i} P(literal j is FALSE)
#
#  Marginals: P(x_i=1) = σ(Σ_a sign_{i,a} × log(warning[a→i] + ε))
#  where sign_{i,a} = +1 if clause a wants x_i=1, else -1.
#
#  Triton kernel: one thread per clause, computes 3 warnings and
#  atomically accumulates them into variable bias buffers.
#  For n=1000, m=4200: 4200 threads — fits a single SM.
#  Runs T=30 iterations with damping α=0.3. Total: ~126K thread blocks.
# ══════════════════════════════════════════════════════════════════════

BP_ITERATIONS = 30
BP_DAMPING    = 0.3     # new = (1-α)×old + α×update
BP_PIN_THRESH = 0.95    # pin variables with |P-0.5| > this/2

if HAS_TRITON:
    @triton.jit
    def bp_clause_update_kernel(
        clauses_v_ptr,   # (m, 3) int32
        clauses_s_ptr,   # (m, 3) int32
        p_ptr,           # (n,) float32 — current marginals
        bias_ptr,        # (n,) float32 — output bias accumulator
        m_val,           # int: number of clauses
        BLOCK: tl.constexpr,
    ):
        """One thread per clause: compute warnings and accumulate biases.

        For clause a = (v0,s0), (v1,s1), (v2,s2):
          q_j = P(literal j is FALSE)
              = 1 - P(v_j=1) if s_j=+1, else P(v_j=1)
          warning to v_j = product of q over other two literals
          bias contribution = s_j × log(warning + ε)
        """
        pid = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < m_val

        # Load clause data (strided access: clauses_v is (m,3) row-major)
        base3 = offs * 3
        v0 = tl.load(clauses_v_ptr + base3 + 0, mask=mask, other=0)
        v1 = tl.load(clauses_v_ptr + base3 + 1, mask=mask, other=0)
        v2 = tl.load(clauses_v_ptr + base3 + 2, mask=mask, other=0)
        s0 = tl.load(clauses_s_ptr + base3 + 0, mask=mask, other=0).to(tl.float32)
        s1 = tl.load(clauses_s_ptr + base3 + 1, mask=mask, other=0).to(tl.float32)
        s2 = tl.load(clauses_s_ptr + base3 + 2, mask=mask, other=0).to(tl.float32)

        # Load current marginals
        p0 = tl.load(p_ptr + v0, mask=mask, other=0.5)
        p1 = tl.load(p_ptr + v1, mask=mask, other=0.5)
        p2 = tl.load(p_ptr + v2, mask=mask, other=0.5)

        # q_j = P(literal j is FALSE)
        # if sign=+1: literal = x_v, FALSE when x_v=0, so q = 1-p
        # if sign=-1: literal = ¬x_v, FALSE when x_v=1, so q = p
        q0 = tl.where(s0 > 0.0, 1.0 - p0, p0)
        q1 = tl.where(s1 > 0.0, 1.0 - p1, p1)
        q2 = tl.where(s2 > 0.0, 1.0 - p2, p2)

        # Clamp to avoid log(0)
        eps = 1e-8

        # Warnings: probability clause NEEDS each variable
        w0 = q1 * q2 + eps
        w1 = q0 * q2 + eps
        w2 = q0 * q1 + eps

        # Bias contributions: sign × log(warning)
        # sign > 0 → clause wants x=1 → positive bias for P(x=1)
        # sign < 0 → clause wants x=0 → negative bias
        b0 = s0 * tl.log(w0)
        b1 = s1 * tl.log(w1)
        b2 = s2 * tl.log(w2)

        # Atomic accumulate into variable bias buffers
        tl.atomic_add(bias_ptr + v0, b0, mask=mask)
        tl.atomic_add(bias_ptr + v1, b1, mask=mask)
        tl.atomic_add(bias_ptr + v2, b2, mask=mask)


def bp_marginals_triton(clauses_v_flat, clauses_s_flat, n, m,
                        n_iter=BP_ITERATIONS, damping=BP_DAMPING):
    """Run BP on factor graph using Triton kernel.

    Args:
        clauses_v_flat: (m*3,) int32 tensor on GPU — flattened variable indices
        clauses_s_flat: (m*3,) int32 tensor on GPU — flattened signs
        n: int, number of variables
        m: int, number of clauses
        n_iter: int, BP iterations
        damping: float, update damping (0=no update, 1=full replacement)

    Returns:
        marginals: (n,) float32 numpy array, P(x_i = 1)
    """
    p = torch.full((n,), 0.5, device=device, dtype=torch.float32)
    BLOCK = 256
    grid = ((m + BLOCK - 1) // BLOCK,)

    for it in range(n_iter):
        bias = torch.zeros(n, device=device, dtype=torch.float32)
        bp_clause_update_kernel[grid](
            clauses_v_flat, clauses_s_flat,
            p, bias, m, BLOCK
        )
        # Sigmoid update with damping
        p_new = torch.sigmoid(bias)
        p = (1.0 - damping) * p + damping * p_new

    return p.cpu().numpy()


# Numba fallback for non-Triton environments
@njit(cache=True)
def bp_marginals_numba(clauses_v, clauses_s, n, m,
                       n_iter=30, damping=0.3):
    """CPU fallback BP using Numba.

    Returns:
        marginals: (n,) float64 array, P(x_i = 1)
    """
    p = np.full(n, 0.5, dtype=np.float64)
    eps = 1e-8

    for iteration in range(n_iter):
        bias = np.zeros(n, dtype=np.float64)

        for c in range(m):
            v0, v1, v2 = clauses_v[c, 0], clauses_v[c, 1], clauses_v[c, 2]
            s0, s1, s2 = clauses_s[c, 0], clauses_s[c, 1], clauses_s[c, 2]

            # q = P(literal is FALSE)
            q0 = (1.0 - p[v0]) if s0 > 0 else p[v0]
            q1 = (1.0 - p[v1]) if s1 > 0 else p[v1]
            q2 = (1.0 - p[v2]) if s2 > 0 else p[v2]

            w0 = q1 * q2 + eps
            w1 = q0 * q2 + eps
            w2 = q0 * q1 + eps

            bias[v0] += s0 * np.log(w0)
            bias[v1] += s1 * np.log(w1)
            bias[v2] += s2 * np.log(w2)

        # Sigmoid update with damping
        for i in range(n):
            p_new = 1.0 / (1.0 + np.exp(-bias[i]))
            p[i] = (1.0 - damping) * p[i] + damping * p_new

    return p.astype(np.float32)


def compute_bp_marginals(helper, n, m):
    """Dispatch to Triton or Numba BP implementation.

    Returns:
        marginals: (n,) float32 numpy array, P(x_i = 1)
    """
    if HAS_TRITON and device.type == 'cuda':
        # Flatten clause arrays for Triton (contiguous int32)
        cv_flat = helper.vars_t.reshape(-1).contiguous().int()
        cs_flat = helper.signs_t.reshape(-1).contiguous().int()
        return bp_marginals_triton(cv_flat, cs_flat, n, m)
    else:
        return bp_marginals_numba(helper.clauses_v, helper.clauses_s, n, m)


def bp_guided_init(marginals, n, K_engines, particles_per_engine,
                   pin_threshold=BP_PIN_THRESH):
    """Initialize gravity particles from BP marginals.

    Strategy:
      - Variables with P > pin_threshold → start at +0.8 × sign
      - Variables with P < 1-pin_threshold → start at -0.8 × sign
      - Others → continuous value 2P-1 + small noise

    Returns:
        s_init: (K_engines, particles, n) float32 tensor
        n_pinned: int, number of variables pinned by BP
    """
    # Convert marginals to continuous [-1, 1]
    s_base = 2.0 * marginals - 1.0   # P=0.5 → 0, P=1 → +1, P=0 → -1

    n_pinned = int(np.sum((marginals > pin_threshold) |
                          (marginals < 1.0 - pin_threshold)))

    # Build tensor: each engine gets same BP init + engine-specific noise
    s_all = np.zeros((K_engines, particles_per_engine, n), dtype=np.float32)
    for k in range(K_engines):
        noise_scale = 0.05 + 0.15 * k / max(1, K_engines - 1)
        for p in range(particles_per_engine):
            s_all[k, p] = s_base + np.random.randn(n).astype(np.float32) * noise_scale
    np.clip(s_all, -0.95, 0.95, out=s_all)

    # Pin high-confidence variables — reduce noise on them
    for i in range(n):
        if marginals[i] > pin_threshold:
            s_all[:, :, i] = 0.85 + np.random.randn(
                K_engines, particles_per_engine).astype(np.float32) * 0.02
        elif marginals[i] < 1.0 - pin_threshold:
            s_all[:, :, i] = -0.85 + np.random.randn(
                K_engines, particles_per_engine).astype(np.float32) * 0.02

    return torch.from_numpy(s_all).to(device), n_pinned


# ══════════════════════════════════════════════════════════════════════
#  BP-GUIDED GRAVITY ROUND (R1 alternative for H22)
#
#  Instead of random diverse seeds → gravity, we:
#   1. Run BP to get marginals
#   2. Initialize particles from marginals
#   3. Run gravity as normal
#  This narrows the search space for gravity by starting near
#  BP's suggested solution.
# ══════════════════════════════════════════════════════════════════════

def run_round_gravity_bp(helpers, n, m, particles, steps, batch_size):
    """R1 gravity with BP-guided initialization.

    Returns:
        engine_results: list of candidate lists per instance
        total_pinned: int, total pinned variables across all instances
    """
    N = len(helpers)
    K = K_ENGINES
    total = N * K
    total_pinned = 0

    all_candidates = [None] * total

    # Compute BP marginals for each instance
    all_marginals = []
    all_s_init = []
    for i in range(N):
        marginals = compute_bp_marginals(helpers[i], n, m)
        s_init, n_pinned = bp_guided_init(marginals, n, K, particles)
        all_marginals.append(marginals)
        all_s_init.append(s_init)  # (K, P, n)
        total_pinned += n_pinned

    # Flatten for batched gravity (same structure as run_round_gravity)
    flat_helper_idx = []
    for i in range(N):
        for k in range(K):
            flat_helper_idx.append(i)

    for bs in range(0, total, batch_size):
        be = min(bs + batch_size, total)
        B = be - bs

        vars_batch = torch.stack([
            helpers[flat_helper_idx[e]].vars_t
            for e in range(bs, be)
        ])
        signs_batch = torch.stack([
            helpers[flat_helper_idx[e]].signs_t
            for e in range(bs, be)
        ])
        pos_mask_batch = torch.stack([
            helpers[flat_helper_idx[e]].pos_mask
            for e in range(bs, be)
        ])

        # Build s from BP-guided initializations
        s_parts = []
        for e in range(bs, be):
            inst_idx = flat_helper_idx[e]
            eng_idx = e - inst_idx * K
            s_parts.append(all_s_init[inst_idx][eng_idx])
        s = torch.stack(s_parts)  # (B, P, n)

        s_final = _gravity_core(s, vars_batch, signs_batch, steps=steps)

        batch_candidates = _fused_extract_candidates(
            s_final, vars_batch, signs_batch, pos_mask_batch,
            top_k=TOP_K_PER_ENGINE
        )
        for b in range(B):
            all_candidates[bs + b] = batch_candidates[b]

        del s, s_final, vars_batch, signs_batch, pos_mask_batch

    result = []
    for i in range(N):
        inst_cands = []
        for k in range(K):
            inst_cands.extend(all_candidates[i * K + k])
        result.append(inst_cands)
    return result, total_pinned


# ══════════════════════════════════════════════════════════════════════
#  PROCESS ONE INSTANCE — H22 version
#
#  Full pipeline:
#   1. Clause-signature clustering (from H21)
#   2. Weighted BPR (H22 cross-clause scoring)
#   3. Conflict-core micro-solver (post-BPR, 1-5 unsat)
#   4. Basin memory tracking (from H21)
# ══════════════════════════════════════════════════════════════════════

def process_instance_round_h22(helper, candidates_list, best_model,
                               best_unsat, n, bpr_max_flips,
                               basin_memory):
    """Process one instance with all H22 upgrades.

    Returns: (solved, model, unsat, n_basins, n_bpr, grav_only,
              n_core_solves)
    """
    m = helper.m
    candidates = candidates_list
    n_core_solves = 0

    # Check gravity-only solves
    for c in candidates:
        if c['unsat'] == 0:
            return True, c['assignment'], 0, 0, 0, True, 0

    # Stage 1: reject if unsat > 20% × n
    threshold = int(UNSAT_REJECT_FRAC * n)
    candidates = [c for c in candidates if c['unsat'] <= threshold]

    if not candidates:
        return False, best_model, best_unsat, 0, 0, False, 0

    # Stage 2: sort by unsat, keep top
    candidates.sort(key=lambda c: c['unsat'])
    candidates = candidates[:TOP_CANDIDATES]

    # Clause-signature clustering (H21)
    n_cand = len(candidates)
    signatures = np.zeros((n_cand, SIG_BUCKETS), dtype=np.float32)
    cand_unsat = np.array([c['unsat'] for c in candidates], dtype=np.int32)
    for i in range(n_cand):
        signatures[i] = clause_signature_numba(
            candidates[i]['assignment'],
            helper.clauses_v, helper.clauses_s,
            SIG_BUCKETS
        )
    centers, n_clusters = basin_cluster_signature_numba(
        signatures, cand_unsat, SIG_TAU_COS, MAX_BASINS
    )
    basin_indices = [int(centers[c]) for c in range(n_clusters)]

    n_bpr_ran = 0
    solved = False
    new_best_model = best_model
    new_best_unsat = best_unsat

    if not basin_indices:
        return False, best_model, best_unsat, 0, 0, False, 0

    # Submit WEIGHTED BPR workers per basin
    futures = {}
    for bi in basin_indices:
        cand = candidates[bi]
        unsat_frac = cand['unsat'] / m
        n_attempts = basin_memory.bpr_attempts_for(unsat_frac)

        for attempt in range(n_attempts):
            f = _BPR_POOL.submit(
                bpr_worker_weighted,          # ← H22: weighted BPR
                helper.clauses_v, helper.clauses_s,
                cand['assignment'].copy(), cand['confidence'],
                bpr_max_flips, 0.5, 0.01, 0.1, BPR_BETA,
                CHAIN_PATIENCE, BRANCH_PATIENCE,
                COOL_RATE, STUB_FRAC, WEIGHT_BUMP, WEIGHT_DECAY
            )
            futures[f] = (bi, unsat_frac)
            n_bpr_ran += 1

    for f in as_completed(futures):
        bi, unsat_frac = futures[f]
        is_sat, flips_used, n_chains, sol = f.result()

        if is_sat:
            basin_memory.record(unsat_frac, True)
            solved = True
            new_best_model = sol
            new_best_unsat = 0
            for remaining in futures:
                if remaining is not f:
                    remaining.cancel()
            break

        sol_unsat = count_unsat(helper.clauses_v, helper.clauses_s, sol)

        # ─── H22: Conflict-core micro-solver on near-misses ───
        if 1 <= sol_unsat <= 5:
            core_solved, core_sol, core_unsat = conflict_core_solver(
                helper.clauses_v, helper.clauses_s, sol
            )
            if core_solved:
                basin_memory.record(unsat_frac, True)
                solved = True
                new_best_model = core_sol
                new_best_unsat = 0
                n_core_solves += 1
                for remaining in futures:
                    if remaining is not f:
                        remaining.cancel()
                break
            elif core_unsat < sol_unsat:
                sol = core_sol
                sol_unsat = core_unsat

        basin_memory.record(unsat_frac, False)

        if sol_unsat < new_best_unsat:
            new_best_unsat = sol_unsat
            new_best_model = sol.copy()

    return (solved, new_best_model, new_best_unsat,
            n_clusters, n_bpr_ran, False, n_core_solves)


# ══════════════════════════════════════════════════════════════════════
#  COMPILE WARMUP — H22 functions
# ══════════════════════════════════════════════════════════════════════

# Warmup conflict_core_solver
_wv22 = np.array([[0, 1, 2], [1, 2, 0]], dtype=np.int32)
_ws22 = np.array([[1, -1, 1], [-1, 1, -1]], dtype=np.int32)
_wa22 = np.array([1, 0, 1], dtype=np.int32)
_ = conflict_core_solver(_wv22, _ws22, _wa22)

# Warmup bpr_chain_weighted
_ww22 = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_chain_weighted(_wv22, _ws22, _wa22, _ww22, max_flips=100,
                       chain_patience=20, branch_patience=5)

# Warmup build_vig_communities
_ = build_vig_communities(_wv22, 3, max_iter=3)

# Warmup BP numba fallback
_ = bp_marginals_numba(_wv22, _ws22, 3, 2, n_iter=2)

print('✓ H22 — Ultimate Basin Search')
print()
print('  NEW components:')
print('    1. Conflict-Core Micro-Solver (Numba)')
print('       → when BPR reaches 1-5 unsat: brute-force 2^k on core vars (k≤15)')
print('    2. Conflict-Degree Weighted BPR (Numba)')
print('       → greedy step boosts variables in multiple violated clauses')
print('    3. Community-Aware Seeding (Numba VIG + label propagation)')
print('       → R2-R5 seeds flip whole structural communities')
print(f'    4. Belief Propagation Warm-Start '
      f'({"Triton GPU" if HAS_TRITON else "Numba CPU"} — '
      f'{BP_ITERATIONS} iterations, α={BP_DAMPING})')
print('       → computes P(x_i=1) marginals, pins high-confidence vars')
print()
print('  PRESERVED from H21:')
print(f'    • Clause-signature clustering (B={SIG_BUCKETS}, τ={SIG_TAU_COS})')
print(f'    • Basin memory ({N_BPR_BOOST}/{N_BPR_BASE}/{N_BPR_MIN} adaptive)')
print(f'    • K={K_ENGINES} engines × {P_R1} ptcl, per-engine noise')
print(f'    • Clean gravity core, all GPU opts [A-F]')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H22 Experiment: Ultimate Basin Search
# ══════════════════════════════════════════════════════════════════════

h15_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 100.0,
    (4.2, 500): 54.0,  (4.2, 750): 36.0,  (4.2, 1000): 24.0,
}
h19 = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 92.0,  (4.0, 750): 96.0,  (4.0, 1000): 80.0,
    (4.2, 500): 22.0,  (4.2, 750): 14.0,  (4.2, 1000):  2.0,
}
h13_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 94.0,  (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 30.0,  (4.2, 750): 32.0,  (4.2, 1000): 16.0,
}
h14_ens = {
    (3.8, 500): 100.0, (3.8, 750): 100.0, (3.8, 1000): 100.0,
    (4.0, 500): 100.0, (4.0, 750): 100.0, (4.0, 1000): 98.0,
    (4.2, 500): 42.0,  (4.2, 750): 28.0,  (4.2, 1000): 20.0,
}

ALPHAS   = [4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50

results_h22 = {}
round_stats_h22 = {}
all_basin_memories_h22 = {}

print('=' * 160)
print('H22 — Ultimate Basin Search')
print(f'  K={K_ENGINES} engines | R1: BP-guided + {STEPS_R1} steps × {P_R1} ptcl | '
      f'R2-R5: community seeds + weighted BPR × {P_RN} ptcl')
print(f'  Signature: B={SIG_BUCKETS} | τ_cos={SIG_TAU_COS} | Max {MAX_BASINS} basins | '
      f'BP: {"Triton" if HAS_TRITON else "Numba"} {BP_ITERATIONS}it | '
      f'Core solver: ≤5 unsat, ≤15 vars')
print('=' * 160)
print(f"  {'α':>5} | {'n':>5} | {'Solved':>6} | {'GravO':>5} | "
      f"{'Core':>5} | {'Rnds':>4} | {'Basins':>6} | {'BPRs':>5} | "
      f"{'Pinned':>6} | {'Comms':>5} | "
      f"{'H15':>5} | {'H19':>5} | {'ΔvsH15':>7} | Time")
print('  ' + '-' * 140)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()

        basin_mem = BasinMemory(n_bins=20)

        # ═══════════════════════════════════════════════════════════
        # Generate all instances
        # ═══════════════════════════════════════════════════════════
        all_instances = [generate_3sat_instance(n_var, m_cls) for _ in range(N_INST)]
        helpers = [InstanceHelper(n_var, all_instances[i]) for i in range(N_INST)]

        # ═══════════════════════════════════════════════════════════
        # PRE-COMPUTE: VIG communities for each instance
        # ═══════════════════════════════════════════════════════════
        t_pre = time.time()
        all_communities = []
        all_n_comms = []
        for i in range(N_INST):
            labels, nc = build_vig_communities(helpers[i].clauses_v, n_var)
            all_communities.append(labels)
            all_n_comms.append(nc)
        avg_comms = np.mean(all_n_comms)
        t_pre_elapsed = time.time() - t_pre

        # ═══════════════════════════════════════════════════════════
        # TRACKING
        # ═══════════════════════════════════════════════════════════
        inst_solved = [False] * N_INST
        inst_round_solved = [0] * N_INST
        inst_best_model = [None] * N_INST
        inst_best_unsat = [m_cls] * N_INST
        inst_gravity_only = [False] * N_INST

        total_grav_only = 0
        total_basins_found = 0
        total_bpr_ran = 0
        total_core_solves = 0
        total_pinned = 0
        per_round_solves = []

        # ═══════════════════════════════════════════════════════════
        # ROUND 1: BP-guided gravity
        # ═══════════════════════════════════════════════════════════
        t_r1 = time.time()
        print(f'    α={alpha}, n={n_var}: R1 — {N_INST} instances, '
              f'BP-guided + K={K_ENGINES} engines × {P_R1} ptcl × {STEPS_R1} steps')

        engine_results, r1_pinned = run_round_gravity_bp(
            helpers, n_var, m_cls,
            particles=P_R1, steps=STEPS_R1,
            batch_size=GRAVITY_BATCH
        )
        total_pinned += r1_pinned
        t_grav_r1 = time.time() - t_r1

        r1_solves = 0
        r1_basins = 0
        r1_bprs = 0
        r1_cores = 0
        for inst in range(N_INST):
            solved, new_model, new_unsat, n_basins, n_bpr, grav_only, n_cores = \
                process_instance_round_h22(
                    helpers[inst], engine_results[inst],
                    inst_best_model[inst], inst_best_unsat[inst],
                    n_var, FLIPS_R1, basin_mem
                )
            r1_basins += n_basins
            r1_bprs += n_bpr
            r1_cores += n_cores

            if solved:
                inst_solved[inst] = True
                inst_round_solved[inst] = 1
                inst_best_model[inst] = new_model
                inst_best_unsat[inst] = 0
                r1_solves += 1
                if grav_only:
                    inst_gravity_only[inst] = True
                    total_grav_only += 1
            else:
                inst_best_model[inst] = new_model if new_model is not None else inst_best_model[inst]
                inst_best_unsat[inst] = new_unsat

        total_basins_found += r1_basins
        total_bpr_ran += r1_bprs
        total_core_solves += r1_cores
        per_round_solves.append(r1_solves)
        total_solved = sum(inst_solved)

        print(f'      → R1: +{r1_solves} solved ({total_solved}/{N_INST}), '
              f'{r1_basins} basins, {r1_bprs} BPRs, {r1_cores} core-solves | '
              f'pinned={r1_pinned}, comms={avg_comms:.0f} | '
              f'grav={t_grav_r1:.0f}s, total={time.time()-t_r1:.0f}s')

        # ═══════════════════════════════════════════════════════════
        # ROUNDS 2-5: Community-seeded + weighted BPR + core solver
        # ═══════════════════════════════════════════════════════════
        for rnd_idx, esc in enumerate(ESCALATION):
            rnd = rnd_idx + 2

            unsolved_idx = [i for i in range(N_INST) if not inst_solved[i]]
            if not unsolved_idx:
                per_round_solves.append(0)
                continue

            sf = esc['seed_frac']
            sn = esc['noise']
            steps_rn = esc['steps']
            flips_rn = esc['flips']

            t_rn = time.time()
            print(f'    α={alpha}, n={n_var}: R{rnd} — {len(unsolved_idx)} unsolved, '
                  f'K={K_ENGINES} engines × {P_RN} ptcl × {steps_rn} steps '
                  f'({int(sf*100)}% seed, σ={sn:.1f}, community seeds)')

            unsolved_helpers = [helpers[i] for i in unsolved_idx]
            global_bests = [inst_best_model[i] for i in unsolved_idx]

            engine_results = run_round_seeded_gravity(
                unsolved_helpers, global_bests, n_var, m_cls,
                particles=P_RN, steps=steps_rn,
                seed_frac=sf, seed_noise=sn,
                batch_size=GRAVITY_BATCH
            )
            t_grav_rn = time.time() - t_rn

            rn_solves = 0
            rn_basins = 0
            rn_bprs = 0
            rn_cores = 0

            for ui_idx, orig_idx in enumerate(unsolved_idx):
                solved, new_model, new_unsat, n_basins, n_bpr, grav_only, n_cores = \
                    process_instance_round_h22(
                        helpers[orig_idx], engine_results[ui_idx],
                        inst_best_model[orig_idx], inst_best_unsat[orig_idx],
                        n_var, flips_rn, basin_mem
                    )
                rn_basins += n_basins
                rn_bprs += n_bpr
                rn_cores += n_cores

                if solved:
                    inst_solved[orig_idx] = True
                    inst_round_solved[orig_idx] = rnd
                    inst_best_model[orig_idx] = new_model
                    inst_best_unsat[orig_idx] = 0
                    rn_solves += 1
                    if grav_only:
                        inst_gravity_only[orig_idx] = True
                        total_grav_only += 1
                else:
                    if new_unsat < inst_best_unsat[orig_idx]:
                        inst_best_model[orig_idx] = new_model
                        inst_best_unsat[orig_idx] = new_unsat
                    elif new_model is not None:
                        inst_best_model[orig_idx] = new_model

            total_basins_found += rn_basins
            total_bpr_ran += rn_bprs
            total_core_solves += rn_cores
            per_round_solves.append(rn_solves)
            total_solved = sum(inst_solved)

            print(f'      → R{rnd}: +{rn_solves} solved ({total_solved}/{N_INST}), '
                  f'{rn_basins} basins, {rn_bprs} BPRs, {rn_cores} core-solves | '
                  f'grav={t_grav_rn:.0f}s, total={time.time()-t_rn:.0f}s')

            if total_solved == N_INST:
                break

        # ═══════════════════════════════════════════════════════════
        # RESULTS for this (α, n)
        # ═══════════════════════════════════════════════════════════
        elapsed = time.time() - t0
        solved_pct = sum(inst_solved) / N_INST * 100
        results_h22[(alpha, n_var)] = solved_pct

        h15e = h15_ens[(alpha, n_var)]
        h19v = h19[(alpha, n_var)]
        delta = solved_pct - h15e

        rds = {
            'rounds_used': len(per_round_solves),
            'per_round': per_round_solves,
            'basins': total_basins_found,
            'bpr_ran': total_bpr_ran,
            'grav_only': total_grav_only,
            'core_solves': total_core_solves,
            'pinned': total_pinned,
            'avg_communities': avg_comms,
            'round_solved': inst_round_solved[:],
            'unsolved_unsats': [inst_best_unsat[i] for i in range(N_INST)
                                if not inst_solved[i]],
        }
        round_stats_h22[(alpha, n_var)] = rds
        all_basin_memories_h22[(alpha, n_var)] = basin_mem

        tag = '★' if solved_pct >= 95 else ('▲' if delta > 2 else
              ('≈' if abs(delta) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {solved_pct:5.1f}% | '
              f'{total_grav_only:5d} | {total_core_solves:5d} | '
              f'{len(per_round_solves):4d} | '
              f'{total_basins_found:6d} | {total_bpr_ran:5d} | '
              f'{total_pinned:6d} | {avg_comms:5.0f} | '
              f'{h15e:4.0f}% | {h19v:4.0f}% | '
              f'{delta:+6.1f}% | {elapsed:.0f}s {tag}')
    print('  ' + '-' * 140)


# ══════════════════════════════════════════════════════════════════════
#  FULL COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 160)
print('FULL COMPARISON — H13 → H14 → H15 → H19 → H22')
print('=' * 160)
print(f"  {'α':>5} | {'n':>5} | {'H13E':>5} | {'H14E':>5} | "
      f"{'H15':>5} | {'H19':>5} | {'H22':>5} | "
      f"{'ΔvsH15':>7} | {'ΔvsH19':>7}")
print('  ' + '-' * 80)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        h22v = results_h22.get(key, 0)
        h15v = h15_ens[key]
        h19v = h19[key]
        delta15 = h22v - h15v
        delta19 = h22v - h19v
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{h13_ens[key]:4.0f}% | {h14_ens[key]:4.0f}% | '
              f'{h15v:4.0f}% | {h19v:4.0f}% | {h22v:4.0f}% | '
              f'{delta15:+6.1f}% | {delta19:+6.1f}%')
    print('  ' + '-' * 80)


# ══════════════════════════════════════════════════════════════════════
#  ROUND ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 160)
print('ROUND ANALYSIS — which rounds solved instances?')
print('=' * 160)
print(f"  {'α':>5} | {'n':>5} | {'R1':>4} | {'R2':>4} | {'R3':>4} | "
      f"{'R4':>4} | {'R5':>4} | {'Fail':>4} | "
      f"{'Basins':>6} | {'BPRs':>5} | {'GravO':>5} | {'Core':>5} | Interpretation")
print('  ' + '-' * 140)

for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats_h22.get(key)
        if not rs:
            continue
        round_counts = [0] * MAX_ROUNDS
        for rnd in rs['round_solved']:
            if rnd > 0:
                round_counts[rnd - 1] += 1
        n_fail = N_INST - sum(round_counts)
        parts = []
        if round_counts[0] > 0:
            parts.append(f'R1={round_counts[0]}')
        rescued = sum(round_counts[1:])
        if rescued > 0:
            parts.append(f'+{rescued} rescued')
        if n_fail > 0:
            parts.append(f'{n_fail} hard')
        if rs.get('core_solves', 0) > 0:
            parts.append(f'{rs["core_solves"]} via core-solver')
        interp = ', '.join(parts) if parts else 'All solved R1'
        rc = round_counts
        print(f'  {alpha:5.1f} | {n_var:5d} | '
              f'{rc[0]:4d} | {rc[1] if len(rc)>1 else 0:4d} | '
              f'{rc[2] if len(rc)>2 else 0:4d} | '
              f'{rc[3] if len(rc)>3 else 0:4d} | '
              f'{rc[4] if len(rc)>4 else 0:4d} | '
              f'{n_fail:4d} | '
              f'{rs["basins"]:6d} | {rs["bpr_ran"]:5d} | '
              f'{rs["grav_only"]:5d} | {rs.get("core_solves",0):5d} | '
              f'{interp}')
    print('  ' + '-' * 140)


# ══════════════════════════════════════════════════════════════════════
#  UNSOLVED ANALYSIS + CORE SOLVER STATS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('UNSOLVED INSTANCE ANALYSIS (H22)')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats_h22.get(key)
        if not rs:
            continue
        unsats = rs['unsolved_unsats']
        if unsats:
            unsats_s = sorted(unsats)
            print(f'  α={alpha}, n={n_var}: {len(unsats)} unsolved | '
                  f'unsat: min={unsats_s[0]}, '
                  f'median={unsats_s[len(unsats_s)//2]}, '
                  f'max={unsats_s[-1]}, '
                  f'mean={np.mean(unsats_s):.1f}')


# ══════════════════════════════════════════════════════════════════════
#  BASIN MEMORY ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('BASIN MEMORY — BPR success rate by unsat-fraction')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        mem = all_basin_memories_h22.get(key)
        if mem:
            print(f'\n  α={alpha}, n={n_var}: '
                  f'{mem.total_samples()} total BPR attempts')
            print(mem.summary())


# ══════════════════════════════════════════════════════════════════════
#  H22 COMPONENT ANALYSIS
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('H22 COMPONENT IMPACT ANALYSIS')
print('=' * 100)
for alpha in ALPHAS:
    for n_var in NS:
        key = (alpha, n_var)
        rs = round_stats_h22.get(key)
        if not rs:
            continue
        solved_pct = results_h22.get(key, 0)
        core = rs.get('core_solves', 0)
        pinned = rs.get('pinned', 0)
        comms = rs.get('avg_communities', 0)
        print(f'  α={alpha}, n={n_var}: {solved_pct:.0f}% solved')
        print(f'    BP warm-start:  {pinned} vars pinned across all instances')
        print(f'    Communities:    {comms:.0f} avg per instance (VIG label-prop)')
        print(f'    Core-solver:    {core} additional solves from brute-force on 1-5 unsat')
        print(f'    Weighted BPR:   cross-clause bonus = 0.15 (conflict hub targeting)')


# ══════════════════════════════════════════════════════════════════════
#  COMPARISON CHART
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating H22 charts...')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('H22 Ultimate Basin Search — Solve Rate by n',
             fontsize=14, fontweight='bold')

all_versions = [
    ('H13E', h13_ens), ('H14E', h14_ens),
    ('H15', h15_ens), ('H19', h19), ('H22', results_h22),
]
colors = ['#888', '#c80', '#08c', '#f44', '#0a0']

for ai, alpha in enumerate(ALPHAS):
    ax = axes[ai]
    for vi, (name, data) in enumerate(all_versions):
        vals = [data.get((alpha, n), 0) for n in NS]
        lw = 3 if name in ('H15', 'H22') else 1.5
        ls = '-' if name in ('H15', 'H19', 'H22') else '--'
        marker = 'o' if name in ('H15', 'H19', 'H22') else '.'
        ax.plot(NS, vals, marker=marker, label=name, color=colors[vi],
                linewidth=lw, linestyle=ls)
    ax.set_title(f'α = {alpha}')
    ax.set_xlabel('n (variables)')
    ax.set_ylabel('Solve Rate (%)')
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('h22_ultimate_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: h22_ultimate_results.png')